# Entrevista de Malvinas — ejecución automática y entrega privada

**En el celular:** abrí este notebook en Colab, tocá **Conectar** y después **Entorno de ejecución → Ejecutar todas** (en algunas versiones aparece como **Ejecutar todo**). Autorizá el acceso con la cuenta de Google que puede abrir los originales. Se utiliza una sola autenticación nativa de Colab y Drive API; no se monta DriveFS. No se solicita la URL, un orden manual ni escribir «SI».

Se detectan los tres archivos conocidos y se produce `ENTREVISTA_MALVINAS_FINAL_AJUSTADA.mp4`, con subtítulos permanentes, y `ENTREVISTA_MALVINAS_SUBTITULOS_AJUSTADOS.srt`. El informe breve y las muestras de las uniones también se guardan en una carpeta privada de Mi unidad. Al finalizar se muestran enlaces reales de Drive, verificados contra tamaño y checksum del servidor.

El orden solicitado y confirmado por el usuario es **≈25 → ≈37 → ≈12 minutos**. Se asignan los tres archivos según las duraciones verificadas con FFprobe. Los checkpoints existentes se reutilizan por identidad del original, y los tiempos del SRT se recalculan para la nueva secuencia. Se conservan las palabras reconocidas y las pausas cortas. Se recortan silencios de al menos 15 segundos solo con audio muy bajo, sin palabras y sin voz detectada, dejando 1,5 segundos en ambos extremos. Los cortes se detallan en el informe para revisión humana. La revisión del texto y la sincronía perceptual siguen pendientes.

La ejecución puede tardar, especialmente sin GPU. Cada intervalo transcrito se guarda en Drive y **Ejecutar todas** retoma automáticamente la misma carpeta si se desconecta. Las verificaciones automáticas y las revisiones humanas pendientes se informan por separado. No se usan APIs pagas ni se publica el material.

**Actualizar este notebook no significa que el video real ya se haya generado.** El procesamiento comienza cuando lo ejecutás en tu Colab autenticado.

Si Google muestra un 403 en la ventana `accounts.google.com` antes de completar la autorización, no es un error de transcripción ni de recursos de GPU. Probá un perfil de Chrome con una sola cuenta autorizada. Este notebook no puede cambiar restricciones de cuenta u organización. Si el error sucede después, la celda indica la etapa y el motivo devuelto por Drive API, sin mostrar credenciales.

La versión ajustada usa texto blanco sobre un recuadro negro semitransparente y se guarda con otro nombre en Drive. La versión anterior permanece disponible. Si el detector de voz falla, se aplica el fondo negro y se conserva todo el metraje, con la limitación indicada en el informe.


In [ ]:
# 1 — Dependencias mínimas. No instala Remotion ni herramientas de diseño.
import sys,subprocess,os,shutil
assert 'google.colab' in sys.modules or os.path.isdir('/content'),'Abrí este notebook en Google Colab.'
subprocess.run([sys.executable,'-m','pip','install','-q','faster-whisper==1.2.1','pysubs2==1.8.0','google-api-python-client>=2,<3'],check=True)
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','ffmpeg','fonts-dejavu-core'],check=True)
filters=subprocess.run(['ffmpeg','-hide_banner','-filters'],capture_output=True,text=True,check=True).stdout
assert ' ass ' in filters and 'loudnorm' in filters,'FFmpeg necesita libass y loudnorm.'
os.environ['HF_HOME']='/content/video_studio_hf'
os.environ['HF_XET_CACHE']='/content/video_studio_hf/xet'
print('Dependencias disponibles. El proceso continuará sin preguntas editoriales.')


In [ ]:
"""Self-contained media engine embedded in the private Colab notebook.
No credentials, original media, transcripts or API calls are embedded here.
"""
from pathlib import Path
from fractions import Fraction
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tempfile
import textwrap

CHECKPOINT_SAVER=None


def run(args):
    p = subprocess.run([str(x) for x in args], capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr[-5000:])
    return p.stdout


def probe(path):
    return json.loads(run(['ffprobe', '-v', 'error', '-show_format', '-show_streams', '-show_data_hash', 'sha256', '-of', 'json', path]))


def checksum(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4*1024**2), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp.json')
    with open(tmp, 'w') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    if CHECKPOINT_SAVER is not None:
        CHECKPOINT_SAVER(path)


def duration(info):
    return float(info['format']['duration'])


def validate(path, expected=None):
    info = probe(path)
    if duration(info) <= 0:
        raise ValueError('Archivo vacío')
    if expected is not None and abs(duration(info)-expected) > .25:
        raise ValueError(f'Duración inesperada: {duration(info)} vs {expected}')
    run(['ffmpeg', '-v', 'error', '-xerror', '-i', path, '-f', 'null', '-'])
    return info


def video_shape(info):
    v = next(s for s in info['streams'] if s['codec_type']=='video')
    w, h = v['width'], v['height']
    rotations = [s.get('rotation', 0) for s in v.get('side_data_list', [])]
    if any(abs(int(r))%180 == 90 for r in rotations):
        w, h = h, w
    return w, h, v.get('avg_frame_rate', v.get('r_frame_rate', '30/1'))


def compatible_video_copy(details):
    signatures=[]
    for info in details:
        v=next(s for s in info['streams'] if s['codec_type']=='video')
        if v['codec_name']!='h264' or abs(float(v.get('start_time',0)))>.05:
            return False
        if any(int(x.get('rotation',0))%360 for x in v.get('side_data_list',[])):
            return False
        if abs(float(v.get('duration',duration(info)))-duration(info))>.1:
            return False
        signatures.append(tuple(v.get(k) for k in ('codec_name','width','height','pix_fmt',
            'avg_frame_rate','r_frame_rate','sample_aspect_ratio','profile','level','extradata_hash')))
    return bool(signatures) and all(s==signatures[0] for s in signatures)


def normalize(source, target, shape, silence=False, copy_video=False):
    target=Path(target); info=probe(source); seconds=duration(info)
    if target.exists():
        return validate(target,seconds)
    w,h,fps=shape
    partial=target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    args=['ffmpeg','-v','error','-n','-i',source]
    audio=any(s['codec_type']=='audio' for s in info['streams']) and not silence
    if not audio:
        args+=['-f','lavfi','-i','anullsrc=r=48000:cl=stereo']
    args+=['-map','0:v:0','-map','0:a:0' if audio else '1:a:0']
    if copy_video:
        args+=['-c:v','copy']
    else:
        args+=['-vf',f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},tpad=stop_mode=clone:stop_duration={seconds}',
               '-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p']
    args+=['-af','aresample=48000:async=1:first_pts=0,afftdn=nr=6:nf=-35,loudnorm=I=-16:TP=-1.5:LRA=11,aresample=48000,apad',
           '-t',str(seconds),'-c:a','aac','-b:a','128k','-ar','48000','-ac','2','-threads','2',
           '-map_metadata','-1','-movflags','+faststart',partial]
    run(args); validate(partial,seconds); os.replace(partial,target)
    return probe(target)


def join_clips(clips, target):
    target = Path(target)
    expected = sum(duration(probe(p)) for p in clips)
    if target.exists():
        return validate(target, expected)
    partial = target.with_suffix('.partial.mp4'); partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as t:
        manifest = Path(t)/'clips.txt'
        manifest.write_text(''.join("file '"+str(Path(p).resolve()).replace("'", "'\\''")+"'\n" for p in clips))
        run(['ffmpeg', '-v', 'error', '-n', '-f', 'concat', '-safe', '0', '-i', manifest,
             '-c', 'copy', '-movflags', '+faststart', partial])
    validate(partial, expected); os.replace(partial,target)
    return probe(target)


def asr_audio_samples(audio):
    """Read the FFmpeg-generated PCM WAV without invoking PyAV's decoder.
    faster-whisper and OpenAI Whisper both accept a 16 kHz float32 NumPy waveform.
    """
    import wave
    import numpy as np
    if isinstance(audio,np.ndarray):
        samples=np.asarray(audio,dtype=np.float32)
        if samples.ndim!=1 or not np.isfinite(samples).all():
            raise ValueError('La señal ASR debe ser mono y contener muestras finitas.')
        return np.ascontiguousarray(samples)
    with wave.open(str(audio),'rb') as wav:
        if (wav.getnchannels(),wav.getframerate(),wav.getsampwidth(),wav.getcomptype())!=(1,16000,2,'NONE'):
            raise ValueError('El intervalo ASR debe ser WAV PCM16 mono a 16000 Hz, generado por FFmpeg.')
        frames=wav.getnframes();raw=wav.readframes(frames)
    if len(raw)!=frames*2:
        raise ValueError('El intervalo WAV está incompleto.')
    return np.frombuffer(raw,dtype='<i2').astype(np.float32)/32768.0


ASR_RUNTIME={}


def load_asr(model_name=None, allow_openai_fallback=True):
    import ctranslate2, gc
    from faster_whisper import WhisperModel
    try:
        available=int(next(x.split()[1] for x in Path('/proc/meminfo').read_text().splitlines() if x.startswith('MemAvailable:')))*1024
    except (OSError,StopIteration,ValueError):
        available=4*1024**3
    gpu=False; compute='int8'
    try:
        gpu=ctranslate2.get_cuda_device_count()>0
        if gpu:
            supported=ctranslate2.get_supported_compute_types('cuda')
            compute='float16' if 'float16' in supported else 'int8_float16'
            gpu=compute in supported
    except (RuntimeError,ValueError):
        gpu=False
    name=model_name or ('small' if gpu or (available>=4*1024**3 and (os.cpu_count() or 1)>=4) else 'base')
    if available<2*1024**3 and not gpu:
        name='base'
    device='cuda' if gpu else 'cpu'
    threads=min(4,os.cpu_count() or 1)
    def cuda_error(error):
        return any(x in str(error).lower() for x in ('cuda','cudnn','cublas','out of memory','insufficient memory'))
    def make_fw(selected):
        return WhisperModel(name,device=selected,compute_type=compute if selected=='cuda' else 'int8',
                           cpu_threads=threads,download_root='/content/video_studio_models')
    ASR_RUNTIME.update(model=name,device=device,engine='faster-whisper',human_review=False)
    try:
        try:
            model=make_fw(device)
        except (RuntimeError,OSError) as error:
            if device!='cuda' or not cuda_error(error):
                raise
            device='cpu'; model=make_fw(device); ASR_RUNTIME['device']=device
        def decode(audio):
            segments,_=model.transcribe(asr_audio_samples(audio),language='es',beam_size=5 if device=='cuda' else 3,
                vad_filter=True,vad_parameters={'threshold':.25,'min_silence_duration_ms':1000,'speech_pad_ms':400},
                word_timestamps=True,condition_on_previous_text=False)
            results=[]
            for s in segments:
                if s.text.strip() and not s.words:
                    raise ValueError('ASR devolvió texto sin marcas temporales de palabra')
                results.append({'text':s.text,'avg_logprob':s.avg_logprob,'no_speech_prob':s.no_speech_prob,
                    'words':[{'word':w.word,'start':w.start,'end':w.end,'probability':w.probability} for w in (s.words or [])]})
            return results
        def infer(audio):
            nonlocal model,device
            try:
                return decode(audio)
            except (RuntimeError,OSError) as error:
                if device!='cuda' or not cuda_error(error):
                    raise
                print('GPU incompatible o sin memoria: se continúa este intervalo en CPU/int8.')
                model=None; gc.collect(); device='cpu'; model=make_fw(device)
                ASR_RUNTIME['device']=device
                return decode(audio)
        return infer,'faster-whisper:'+name
    except Exception as error:
        if not allow_openai_fallback:
            raise
        print('Faster-whisper no se pudo cargar:',type(error).__name__)
        print('Se intenta OpenAI Whisper local, con pesos oficiales y SHA-256 de su cargador. Sin API paga.')
        run([os.sys.executable,'-m','pip','install','-q','openai-whisper==20250625'])
        import whisper
        try:
            import torch
            device='cuda' if torch.cuda.is_available() else 'cpu'
        except ImportError:
            device='cpu'
        try:
            model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
        except (RuntimeError,OSError) as error:
            if device!='cuda' or not cuda_error(error):
                raise
            device='cpu';model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
        ASR_RUNTIME.update(engine='openai-whisper',device=device)
        def infer(audio):
            nonlocal model,device
            try:
                result=model.transcribe(asr_audio_samples(audio),language='es',fp16=device=='cuda',verbose=False,
                                        word_timestamps=True,condition_on_previous_text=False)
            except (RuntimeError,OSError) as error:
                if device!='cuda' or not cuda_error(error):
                    raise
                model=None;gc.collect();device='cpu'
                model=whisper.load_model(name,device=device,download_root='/content/video_studio_models')
                ASR_RUNTIME['device']=device
                result=model.transcribe(asr_audio_samples(audio),language='es',fp16=False,verbose=False,
                                        word_timestamps=True,condition_on_previous_text=False)
            return result['segments']
        return infer,'openai-whisper:'+name


def transcribe_clip(source, source_key, checkpoint_root, infer, engine, chunk=300):
    """2 s context around each owned interval; assign words by midpoint.
    Saves each completed interval through the authenticated Drive API.
    """
    checkpoint_root = Path(checkpoint_root); checkpoint_root.mkdir(parents=True, exist_ok=True)
    seconds = duration(probe(source)); words = []; flags = []
    for i in range(math.ceil(seconds/chunk)):
        owned_start, owned_end = i*chunk, min((i+1)*chunk, seconds)
        start, end = max(0, owned_start-2), min(seconds, owned_end+2)
        key = {'version':3, 'source':source_key, 'engine':engine, 'interval':[owned_start, owned_end]}
        record = checkpoint_root/f'{i:04d}.json'
        reused=record.exists()
        if not reused and globals().get('CHECKPOINTS_ONLY',False):
            raise RuntimeError(f'Falta el checkpoint {record.name} de {checkpoint_root.name}; no se reinició la transcripción.')
        if reused:
            data = json.loads(record.read_text())
            if data['key'] != key:
                raise ValueError('Checkpoint incompatible. Usar otra carpeta de trabajo.')
        else:
            with tempfile.TemporaryDirectory() as t:
                audio = Path(t)/'audio.wav'
                run(['ffmpeg', '-v', 'error', '-n', '-ss', str(start), '-i', source,
                     '-t', str(end-start), '-vn', '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', audio])
                results = infer(audio)
            chosen = []; warnings = []
            for s in results:
                if s.get('avg_logprob', 0)<-.8 or s.get('no_speech_prob', 0)>.5:
                    warnings.append({'start':start+s['words'][0]['start'] if s.get('words') else start,
                                     'text':s.get('text',''), 'reason':'revisar_confianza'})
                for word in s.get('words', []):
                    a, b = start+float(word['start']), start+float(word['end'])
                    if owned_start <= (a+b)/2 < owned_end:
                        chosen.append({'word':word['word'], 'start':max(0,a), 'end':min(seconds,b),
                                       'probability':word.get('probability')})
            data = {'key':key, 'words':chosen, 'review':warnings}
            atomic_json(record, data)
        words += data['words']; flags += data['review']
        status='RECUPERADO, sin transcribir' if reused else 'guardado'
        print(f'{Path(source).name}: intervalo {i+1}/{math.ceil(seconds/chunk)} {status}')
    return words, flags


def cues_from_words(words, width=1920, height=1080):
    line_limit=max(18,min(42,round(42*(width/height)/(16/9))))
    # A recognizer can return several whitespace-separated words in one timed token.
    # Split only that token for layout; its original interval and text are preserved.
    units=[]
    for original in words:
        pieces=original['word'].split()
        if not pieces:continue
        if len(pieces)==1:
            units.append({**original,'word':pieces[0]})
        else:
            total=sum(len(piece) for piece in pieces);cursor=0
            start,end=float(original['start']),float(original['end'])
            for piece in pieces:
                a=start+(end-start)*cursor/total;cursor+=len(piece)
                b=start+(end-start)*cursor/total
                units.append({**original,'word':piece,'start':a,'end':b})
    def text(group):
        return re.sub(r'\s+([,.;:!?])',r'\1',' '.join(w['word'].strip() for w in group)).strip()
    def wrapped(group):
        return textwrap.wrap(text(group),line_limit,break_long_words=False,break_on_hyphens=False)
    groups=[];current=[]
    for word in units:
        # Character count alone can wrap onto THREE lines at word boundaries.
        # Test the actual wrapped candidate before adding the next timed word.
        if current and (len(wrapped(current+[word]))>2 or
                        word['end']-current[0]['start']>5 or
                        word['start']-current[-1]['end']>.7 or
                        (text(current).endswith(('.', '?', '!', ';')) and
                         current[-1]['end']-current[0]['start']>1.2)):
            groups.append(current);current=[]
        current.append(word)
    if current:groups.append(current)
    cues=[]
    for group in groups:
        start,end=max(0,group[0]['start']),max(w['end'] for w in group)
        lines=wrapped(group)
        if cues:start=max(start,cues[-1]['end'])
        end=max(start+.05,end)
        cues.append({'start':start,'end':end,'text':'\n'.join(lines)})
    return cues


def write_subtitles(cues, srt, ass, width, height, box_alpha=32):
    import pysubs2
    subs = pysubs2.SSAFile()
    for cue in cues:
        subs.append(pysubs2.SSAEvent(start=round(cue['start']*1000), end=round(cue['end']*1000),
                                   text=cue['text'].replace('\n',r'\N')))
    subs.info.update(PlayResX=str(width), PlayResY=str(height), WrapStyle='0')
    subs.styles['Default'] = pysubs2.SSAStyle(fontname='DejaVu Sans', fontsize=max(14,48*height/1080),
        primarycolor=pysubs2.Color(255,255,255), outlinecolor=pysubs2.Color(0,0,0,box_alpha),
        backcolor=pysubs2.Color(0,0,0,box_alpha),borderstyle=3,
        outline=max(2,6*height/1080), shadow=0, alignment=pysubs2.Alignment.BOTTOM_CENTER,
        marginl=round(width*.07), marginr=round(width*.07), marginv=round(height*.06))
    subs.save(str(srt)); subs.save(str(ass))


def frame(path, at, target):
    run(['ffmpeg','-v','error','-y','-ss',str(at),'-i',path,'-frames:v','1',target])


def pipeline(sources, local_root, durable_root, infer, engine, intro=None, reuse=None):
    """sources are automatically ordered; originals remain untouched; human review stays pending."""
    local_root = Path(local_root); durable_root = Path(durable_root)
    local_root.mkdir(parents=True,exist_ok=True); durable_root.mkdir(parents=True,exist_ok=True)
    source_hashes = [checksum(p) for p in sources]
    details = [probe(p) for p in sources]
    if len(sources)!=3 or len(set(source_hashes))!=3:
        raise ValueError('Se requieren tres archivos distintos')
    shape = tuple(reuse['shape']) if reuse else video_shape(details[0])
    shape = (shape[0]+shape[0]%2, shape[1]+shape[1]%2, shape[2])
    if shape[2] in ('0/0','0/1'):
        shape = (shape[0],shape[1],'30/1')
    intro_hash = checksum(intro) if intro else None
    key = {'sources':source_hashes,'shape':list(shape),'intro':intro_hash,'engine':engine,'version':3}
    lockfile = durable_root/'job.json'
    if lockfile.exists() and json.loads(lockfile.read_text())!=key:
        raise ValueError('Otro orden/configuración ya usa esta carpeta. Elegir una nueva carpeta de trabajo.')
    atomic_json(lockfile,key)
    required = sum(Path(p).stat().st_size for p in sources)*4 + 1024**3
    if shutil.disk_usage(local_root).free < required:
        raise RuntimeError(f'Espacio local insuficiente. Estimación mínima: {required/1024**3:.1f} GiB')
    copy_video=compatible_video_copy(details)
    normalized = []; rows = []; all_words = []; review = []; offset = 0
    if intro:
        p = local_root/'intro.mp4'; d = normalize(intro,p,shape,silence=True)
        normalized.append(p); offset = duration(d)
    for i, (source, info, sha) in enumerate(zip(sources,details,source_hashes)):
        asset=reuse['assets'][sha] if reuse else None
        cached=Path(asset['normalized']) if asset else None
        if cached is not None and cached.exists():
            target=cached;d=validate(target,duration(info))
            print(f'VIDEO REUTILIZADO: {Path(source).name}, {duration(d)/60:.2f} min')
        else:
            target=local_root/f'normalized_{i}.mp4'
            d=normalize(source,target,shape,copy_video=copy_video)
        normalized.append(target)
        rows.append({'source':Path(source).name,'sha256':sha,'source_duration':duration(info),
                     'output_start':offset,'output_end':offset+duration(d)})
        checkpoint_root=Path(asset['checkpoints']) if asset else durable_root/f'checkpoints_{i}'
        words, warnings = transcribe_clip(target,sha,checkpoint_root,infer,engine)
        all_words += [{**w,'start':w['start']+offset,'end':w['end']+offset} for w in words]
        review += [{**w,'start':w['start']+offset} for w in warnings]
        offset += duration(d)
    joined=local_root/'joined.mp4';join_clips(normalized,joined)
    atomic_json(durable_root/'words_source_timeline.json',all_words)
    base={'preliminary_video_stream_copy':copy_video,'reused_source_checkpoints':bool(reuse),'review':review}
    return render_ready_interview(joined,all_words,rows,local_root,durable_root,base)



def automatic_order(paths):
    from itertools import permutations
    if len(paths)!=3:raise ValueError('Se requieren los tres originales conocidos.')
    info={str(p):probe(p) for p in paths}
    requested=(25*60,37*60,12*60)
    choices=[]
    for order in permutations(paths):
        differences=[abs(duration(info[str(path)])-target) for path,target in zip(order,requested)]
        choices.append((sum(differences),tuple(order),differences))
    choices.sort(key=lambda row:row[0])
    cost,order,differences=choices[0]
    if max(differences)>180 or abs(choices[1][0]-cost)<1:
        raise ValueError('Las duraciones no identifican de forma única los archivos de aproximadamente 25, 37 y 12 minutos. No se cambia el orden por suposición.')
    evidence=[{'source':Path(path).name,'duration':duration(info[str(path)]),
               'minutes':round(duration(info[str(path)])/60,3),'shape':video_shape(info[str(path)]),
               'codecs':[stream.get('codec_name') for stream in info[str(path)]['streams']]} for path in order]
    report={'order':[Path(path).name for path in order],
        'method':'orden indicado por el usuario: aproximadamente 25 → 37 → 12 minutos; archivos identificados por FFprobe',
        'requested_minutes':[25,37,12],'actual_minutes':[row['minutes'] for row in evidence],
        'order_confirmed_by_user':True,'reviewed_by_user':True,'human_review_scope':'solo el orden solicitado',
        'semantic_continuity_verified':False,
        'status':'orden confirmado por el usuario; revisión de texto y sincronía pendiente',
        'evidence':evidence}
    return list(order),report


def prepare_reorder_reuse(sources,base_local,base_work,engine):
    base_local=Path(base_local);base_work=Path(base_work)
    job=base_work/'job.json'
    if not job.exists():return None
    previous=json.loads(job.read_text())
    wanted=[checksum(path) for path in sources]
    old=previous['sources']
    if set(wanted)!=set(old) or len(set(old))!=3:
        raise ValueError('Los originales no coinciden con el trabajo guardado. No se mezclan transcripciones de otros archivos.')
    if previous['engine']!=engine:
        raise ValueError('El motor de los checkpoints no coincide. No se inicia una nueva transcripción.')
    if wanted==old:return None
    assets={sha:{'normalized':str(base_local/f'normalized_{i}.mp4'),
                 'checkpoints':str(base_work/f'checkpoints_{i}')} for i,sha in enumerate(old)}
    # A distinct namespace prevents an existing joined/final MP4 from reusing the old order.
    suffix='orden_25_37_12_'+hashlib.sha256('|'.join(wanted).encode()).hexdigest()[:10]
    return {'shape':previous['shape'],'assets':assets,'namespace':suffix,
            'old_source_order':old,'new_source_order':wanted}


POLISH_OPTIONS={'min_silence':15.0,'threshold_db':-42.0,'padding':1.5,'word_guard':1.0,'box_alpha':32,'version':1}


def merge_spans(spans,total):
    out=[]
    for a,b in sorted((max(0,float(a)),min(total,float(b))) for a,b in spans):
        if b<=a:continue
        if out and a<=out[-1][1]+1e-6:out[-1][1]=max(out[-1][1],b)
        else:out.append([a,b])
    return out


def subtract_spans(spans,protected,total):
    protected=merge_spans(protected,total);out=[]
    for a,b in merge_spans(spans,total):
        cursor=a
        for c,d in protected:
            if d<=cursor:continue
            if c>=b:break
            if c>cursor:out.append([cursor,min(c,b)])
            cursor=max(cursor,d)
            if cursor>=b:break
        if cursor<b:out.append([cursor,b])
    return out


def detect_silences(joined,minimum,threshold_db):
    proc=subprocess.run(['ffmpeg','-hide_banner','-nostats','-i',str(joined),'-vn',
        '-af',f'silencedetect=noise={threshold_db}dB:d={minimum}','-f','null','-'],capture_output=True,text=True)
    if proc.returncode:raise RuntimeError(proc.stderr[-3000:])
    spans=[];start=None;total=duration(probe(joined))
    for line in proc.stderr.splitlines():
        a=re.search(r'silence_start:\s*(-?[0-9.]+)',line)
        b=re.search(r'silence_end:\s*(-?[0-9.]+)',line)
        if a:start=float(a.group(1))
        if b and start is not None:spans.append([start,float(b.group(1))]);start=None
    if start is not None:spans.append([start,total])
    return merge_spans(spans,total)


def detect_voice_in_gaps(joined,gaps):
    from faster_whisper.vad import get_speech_timestamps,VadOptions
    protected=[]
    options=VadOptions(threshold=.25,min_speech_duration_ms=100,min_silence_duration_ms=500,speech_pad_ms=500)
    for number,(a,b) in enumerate(gaps,1):
        print(f'Comprobando ausencia de voz en silencio {number}/{len(gaps)}; no se ejecuta Whisper.')
        cursor=a
        while cursor<b:
            start=max(a,cursor-1);end=min(b,cursor+61)
            with tempfile.TemporaryDirectory() as directory:
                wav=Path(directory)/'vad.wav'
                run(['ffmpeg','-v','error','-n','-ss',str(start),'-i',joined,'-t',str(end-start),
                     '-vn','-ac','1','-ar','16000','-c:a','pcm_s16le',wav])
                samples=asr_audio_samples(wav)
                for speech in get_speech_timestamps(samples,options):
                    protected.append([start+speech['start']/16000,start+speech['end']/16000])
            cursor+=60
    return protected


def build_silence_plan(total,raw_silences,words,voices,settings):
    guard=settings['word_guard'];pad=settings['padding'];minimum=settings['min_silence']
    protected=[[float(w['start'])-guard,float(w['end'])+guard] for w in words if w['word'].strip()]
    protected += [[a-guard,b+guard] for a,b in voices]
    gaps=subtract_spans(raw_silences,protected,total)
    cuts=merge_spans([[a+pad,b-pad] for a,b in gaps if b-a>=minimum and b-a>2*pad],total)
    keep=subtract_spans([[0,total]],cuts,total)
    if not keep:raise ValueError('El plan no puede eliminar todo el video.')
    for word in words:
        if word['word'].strip() and any(a<float(word['end']) and b>float(word['start']) for a,b in cuts):
            raise ValueError('Un corte cruzaría una palabra reconocida; no se aplica.')
    return cuts,keep


def edited_time(t,cuts):
    return max(0,float(t)-sum(max(0,min(float(t),b)-a) for a,b in cuts if float(t)>a))


def source_time(t,keep):
    offset=0
    for a,b in keep:
        if float(t)<offset+b-a-1e-9:return a+max(0,float(t)-offset)
        offset+=b-a
    return keep[-1][1]


def remap_words(words,cuts):
    result=[]
    for word in words:
        if word['word'].strip() and any(a<float(word['end']) and b>float(word['start']) for a,b in cuts):
            raise ValueError('El recorte eliminaría una palabra reconocida.')
        result.append({**word,'start':edited_time(word['start'],cuts),'end':edited_time(word['end'],cuts)})
    return result


def export_burned(joined,ass,final,keep=None):
    final=Path(final);expected=sum(b-a for a,b in keep) if keep is not None else duration(probe(joined))
    if final.exists():return validate(final,expected)
    partial=final.with_suffix('.partial.mp4');partial.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as directory:
        safe=Path(directory)/'captions.ass';shutil.copy2(ass,safe)
        if keep is None:
            args=['ffmpeg','-v','error','-n','-i',joined,'-vf',f'ass={safe}',
                  '-c:a','copy']
        else:
            manifest=Path(directory)/'edit.ffconcat'
            escaped=str(Path(joined).resolve()).replace("'","'\\''")
            lines=['ffconcat version 1.0']
            for a,b in keep:
                lines += ["file '"+escaped+"'",f'inpoint {a:.6f}',f'outpoint {b:.6f}',f'duration {b-a:.6f}']
            manifest.write_text('\n'.join(lines)+'\n')
            # Official concatdec_select drops keyframe pre-roll without deleting spoken frames.
            # Packet timestamps already refer to the shortened timeline, so ASS sees remapped times.
            args=['ffmpeg','-v','error','-n','-copyts','-segment_time_metadata','1','-f','concat','-safe','0','-i',manifest,
                  '-vf',f'select=concatdec_select,ass={safe}',
                  '-af','aselect=concatdec_select,aresample=async=1:first_pts=0',
                  '-fps_mode','passthrough','-c:a','aac','-b:a','128k','-t',f'{expected:.6f}']
        args += ['-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p','-threads','2',
                 '-map_metadata','-1','-movflags','+faststart',partial]
        command=[str(x) for x in args]
        command[1:1]=['-nostdin','-progress','pipe:1','-nostats']
        with tempfile.TemporaryFile(mode='w+t') as errors:
            process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=errors,text=True,bufsize=1)
            last=-5
            try:
                for line in process.stdout:
                    if line.startswith('out_time_us='):
                        try:percent=max(0,min(99,int(int(line.split('=',1)[1])/1000000/expected*100)))
                        except ValueError:continue
                        if percent>=last+5:
                            print(f'Exportación ajustada: {percent} %',flush=True);last=percent
                status=process.wait()
            except BaseException:
                process.terminate();process.wait();raise
            finally:
                process.stdout.close()
            if status:
                errors.seek(0);raise RuntimeError(errors.read()[-5000:])
        print('Exportación: 100 %. Verificando decodificación completa.',flush=True)
        validate(partial,expected);os.replace(partial,final)
    return probe(final)


def render_ready_interview(joined,source_words,source_clips,local_root,durable_root,base=None,settings=None):
    settings=dict(settings or POLISH_OPTIONS);base=dict(base or {})
    total=duration(probe(joined))
    signature={'media':checksum(joined),'words':hashlib.sha256(json.dumps(source_words,sort_keys=True).encode()).hexdigest(),
               'settings':settings}
    edit_id=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:12]
    local=Path(local_root)/('ajustes_'+edit_id);durable=Path(durable_root)/('ajustes_'+edit_id)
    local.mkdir(parents=True,exist_ok=True);durable.mkdir(parents=True,exist_ok=True)
    master=Path(durable_root)/'words_source_timeline.json'
    atomic_json(master,source_words)
    plan_path=durable/'edit_plan.json'
    if plan_path.exists():
        plan=json.loads(plan_path.read_text())
        if plan['signature']!=signature:raise ValueError('El plan guardado pertenece a otro material.')
        cuts=plan['cuts'];keep=plan['keep']
    else:
        print('Analizando silencios largos; se conserva la transcripción existente.')
        raw=detect_silences(joined,settings['min_silence'],settings['threshold_db'])
        word_blocks=[[w['start']-settings['word_guard'],w['end']+settings['word_guard']] for w in source_words if w['word'].strip()]
        candidates=[p for p in subtract_spans(raw,word_blocks,total) if p[1]-p[0]>=settings['min_silence']]
        voices=[];vad_error=None
        try:
            voices=detect_voice_in_gaps(joined,candidates) if candidates else []
            cuts,keep=build_silence_plan(total,raw,source_words,voices,settings)
        except Exception as error:
            # A secondary detector failure cannot justify deleting uncertain speech.
            vad_error=type(error).__name__;cuts=[];keep=[[0,total]]
            print('No se verificó ausencia de voz: se conserva el metraje y se aplica solo el fondo negro.')
        plan={'signature':signature,'source_seconds':total,'raw_silences':raw,'vad_voice_spans':voices,
              'vad_checked':bool(candidates) and vad_error is None,'vad_error':vad_error,'cuts':cuts,'keep':keep}
        atomic_json(plan_path,plan)
    cuts=merge_spans(cuts,total)
    expected_keep=subtract_spans([[0,total]],cuts,total)
    if expected_keep!=keep:raise ValueError('El plan de cortes guardado es inconsistente.')
    adjusted=remap_words(source_words,cuts)
    if len(adjusted)!=len(source_words):raise ValueError('No se conserva la transcripción completa.')
    width,height,_=video_shape(probe(joined))
    cues=cues_from_words(adjusted,width,height)
    if not cues:raise ValueError('No hay subtítulos para la versión ajustada.')
    expected=sum(b-a for a,b in keep)
    if cues[-1]['end']>expected+.1:raise ValueError('Subtítulos fuera de la nueva duración.')
    srt=local/'ENTREVISTA_MALVINAS_SUBTITULOS_AJUSTADOS.srt'
    ass=local/'captions_black_box.ass';write_subtitles(cues,srt,ass,width,height,box_alpha=settings['box_alpha'])
    shutil.copy2(srt,durable/srt.name)
    words_file=durable/'words.json';atomic_json(words_file,adjusted)
    final=local/'ENTREVISTA_MALVINAS_FINAL_AJUSTADA.mp4'
    print(f'Recortes: {len(cuts)}; silencio retirado: {sum(b-a for a,b in cuts):.1f} s. Exportando una versión ajustada.')
    final_info=export_burned(joined,ass,final,keep=keep if cuts else None)
    streams=final_info['streams']
    assert next(s for s in streams if s['codec_type']=='video')['codec_name']=='h264'
    assert next(s for s in streams if s['codec_type']=='audio')['codec_name']=='aac'
    rows=[]
    for row in source_clips:
        rows.append({**row,'output_start':edited_time(row['output_start'],cuts),
                     'output_end':edited_time(row['output_end'],cuts)})
    review=[]
    for warning in base.get('review',[]):
        if warning.get('reason')=='velocidad_lectura':continue
        warning=dict(warning)
        if 'start' in warning:
            warning['source_start']=warning.get('source_start',warning['start'])
            warning['start']=edited_time(warning['source_start'],cuts)
        review.append(warning)
    for i,cue in enumerate(cues,1):
        cps=len(cue['text'].replace('\n',''))/(cue['end']-cue['start'])
        if cps>22:review.append({'cue':i,'start':cue['start'],'reason':'velocidad_lectura','cps':round(cps,1)})
    cut_rows=[{'source_start':a,'source_end':b,'removed_seconds':b-a,'new_time':edited_time(a,cuts),
        'sources':[row['source'] for row in source_clips if row['output_start']<b and row['output_end']>a]} for a,b in cuts]
    report={**base,'clips':rows,'source_clips':source_clips,'intro_seconds':edited_time(source_clips[0]['output_start'],cuts),
        'source_seconds':total,'expected_seconds':expected,'final_seconds':duration(final_info),'decode_complete':True,
        'subtitles_count':len(cues),'human_sync_review':False,'human_editorial_review':False,'review':review,
        'source_joined_file':str(joined),'source_words_file':str(master),'words_file':str(words_file),
        'source_word_count':len(source_words),'output_word_count':len(adjusted),'recognized_words_preserved':True,
        'subtitle_style':'texto blanco sobre recuadro negro semitransparente',
        'silence_edit':{'settings':settings,'cuts':cut_rows,'removed_seconds':sum(b-a for a,b in cuts),
                        'vad_checked':plan['vad_checked'],'vad_error':plan['vad_error'],'human_review':False},
        'final_sha256':checksum(final),'srt_sha256':checksum(srt)}
    import numpy as np
    shots=[]
    for index in sorted(set([0,len(cues)//4,len(cues)//2,3*len(cues)//4,len(cues)-1])):
        cue=cues[index];at=(cue['start']+cue['end'])/2;old_at=source_time(at,keep)
        def pixels(path,when):
            return subprocess.run(['ffmpeg','-v','error','-ss',str(when),'-i',str(path),'-frames:v','1',
                '-f','rawvideo','-pix_fmt','gray','-'],check=True,capture_output=True).stdout
        before=np.frombuffer(pixels(joined,old_at),dtype=np.uint8).astype(np.int16)
        after=np.frombuffer(pixels(final,at),dtype=np.uint8).astype(np.int16)
        if before.size!=after.size:raise ValueError('No coincide la resolución para comprobar subtítulos.')
        changed=int(np.count_nonzero(np.abs(after-before)>35))
        if changed<10:raise ValueError('No se detectó el recuadro/subtítulo en el fotograma esperado.')
        shot=local/f'check_{index:04d}.png';frame(final,at,shot)
        shots.append({'at':at,'source_at':old_at,'file':str(shot),'cue':cue['text'],'changed_pixels':changed})
    report['subtitle_frame_checks']=shots
    atomic_json(durable/'validation.json',report)
    return final,srt,report


In [ ]:
# 2 — Una sola autenticación nativa de Colab. No se monta DriveFS.
# Si Google rechaza el consentimiento, no se intenta eludir su control de acceso.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
import google.auth
from datetime import datetime
from pathlib import PurePosixPath
from googleapiclient.errors import HttpError

FOLDER_ID='1ps24ehNBibgSXvSNF-h8JSz2MYZv4BGX'
EXPECTED=['VID-20261107-WA0034.mp4','VID-20261107-WA0035.mp4','VID-20261107-WA0036.mp4']
print('AUTORIZACIÓN: Colab solicitará acceso con tu cuenta de Google.')
try:
    auth.authenticate_user()
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
except Exception as error:
    raise RuntimeError('No se completó la autorización de Google en Colab. Si la ventana accounts.google.com muestra 403, probá un perfil de Chrome con una sola cuenta que tenga acceso a los videos. No cambies los archivos a públicos ni compartas tokens.') from error

def explain_drive_error(error,stage):
    status=getattr(getattr(error,'resp',None),'status',None)
    reason='sin detalle'
    try:
        data=json.loads(error.content.decode()).get('error',{})
        reasons=[x.get('reason','') for x in data.get('errors',[])]
        reasons += [x.get('reason','') for x in data.get('details',[]) if isinstance(x,dict)]
        reason=', '.join(x for x in reasons if x) or data.get('status','sin detalle')
    except (ValueError,AttributeError,UnicodeError):
        pass
    help_text=''
    if status==403 and any(x in reason for x in ('insufficientPermissions','ACCESS_TOKEN_SCOPE_INSUFFICIENT')):
        help_text=' La autorización no incluye el acceso requerido a Drive; volver a autorizar Colab con la cuenta correcta.'
    elif status==403 and any(x in reason.lower() for x in ('domain','admin','policy','restricted')):
        help_text=' Google indica una restricción de la cuenta u organización; corresponde revisar esa política con su administrador.'
    elif status in (403,404):
        help_text=' Confirmá que la misma cuenta autorizada puede abrir la carpeta fuente en drive.google.com. Un 404 también puede indicar falta de acceso, no necesariamente que la carpeta no exista.'
    return RuntimeError(f'Drive API: HTTP {status}; etapa {stage}; motivo {reason}.'+help_text)

# Se distingue el rechazo de Google en el consentimiento de un error posterior de la API.
try:
    source_folder=service.files().get(fileId=FOLDER_ID,supportsAllDrives=True,fields='id,mimeType').execute()
    assert source_folder['mimeType']=='application/vnd.google-apps.folder','El ID fuente no identifica una carpeta.'
except HttpError as error:
    raise explain_drive_error(error,'comprobar acceso a la carpeta fuente') from None
print('AUTORIZACIÓN COMPLETA: la API puede leer la carpeta fuente.')
LOCAL=Path('/content/video_studio_malvinas');LOCAL.mkdir(exist_ok=True)
INPUT=LOCAL/'input';INPUT.mkdir(exist_ok=True)
items=[];token=None
while True:
    page=service.files().list(q=f"'{FOLDER_ID}' in parents and trashed=false",pageToken=token,
        pageSize=1000,includeItemsFromAllDrives=True,supportsAllDrives=True,
        fields='nextPageToken,files(id,name,size,md5Checksum,mimeType,capabilities/canDownload)').execute()
    items+=page.get('files',[]);token=page.get('nextPageToken')
    if not token:break
selected=[]
for name in EXPECTED:
    matches=[f for f in items if f['name']==name]
    assert len(matches)==1,f'Bloqueo real: falta el original o hay duplicados de {name}.'
    info=matches[0]
    assert info.get('capabilities',{}).get('canDownload',True),f'Sin permiso de descarga: {name}'
    assert int(info.get('size',0))>0,f'Archivo vacío/no descargable: {name}'
    selected.append(info)

# La misma versión de los originales recupera sus checkpoints, sin pedir nombres o rutas.
fingerprint=hashlib.sha256(json.dumps([(f['id'],f['size'],f.get('md5Checksum')) for f in selected]).encode()).hexdigest()[:20]
JOB_TAG='malvinas_auto_v3_'+fingerprint
query="'root' in parents and trashed=false and mimeType='application/vnd.google-apps.folder' and appProperties has { key='videoStudioJob' and value='"+JOB_TAG+"' }"
candidates=service.files().list(q=query,fields='files(id,name)',orderBy='createdTime desc',pageSize=100).execute().get('files',[])
def private_owner_only(file_id):
    permissions=service.permissions().list(fileId=file_id,fields='permissions(type,role)').execute().get('permissions',[])
    return bool(permissions) and all(p['type']=='user' and p['role']=='owner' for p in permissions)
remote_folder=next((f for f in candidates if private_owner_only(f['id'])),None)
if remote_folder is None:
    name='VideoStudioAI_Malvinas_'+datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_folder=service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder',
        'appProperties':{'videoStudioJob':JOB_TAG}},fields='id,name').execute()
assert private_owner_only(remote_folder['id']),'Bloqueo real: no se pudo disponer de una carpeta privada.'
# Caché local restaurada desde Drive; no necesita DriveFS ni un segundo consentimiento.
DURABLE=LOCAL/'checkpoint_cache'/fingerprint
DURABLE.mkdir(parents=True,exist_ok=True)


def upload_private_verified(path,mimetype,remote_name=None,checkpoint_relative=None):
    path=Path(path);name=remote_name or path.name
    escaped=name.replace('\\','\\\\').replace("'","\\'")
    matches=service.files().list(q=f"'{remote_folder['id']}' in parents and name='{escaped}' and trashed=false",
        fields='files(id,size,md5Checksum)',pageSize=100).execute().get('files',[])
    assert len(matches)<=1,'Hay copias ambiguas de un archivo generado; no se modifica ninguna sin identificarlas.'
    local_md5=checksum(path,'md5');result=None
    if matches and matches[0].get('md5Checksum')==local_md5 and int(matches[0]['size'])==path.stat().st_size:
        result=matches[0]
    else:
        media=MediaFileUpload(str(path),mimetype=mimetype,chunksize=8*1024**2,resumable=True)
        if matches:
            request=service.files().update(fileId=matches[0]['id'],media_body=media,fields='id')
        else:
            body={'name':name,'parents':[remote_folder['id']]}
            if checkpoint_relative is not None:body['appProperties']={'vsaiRelPath':checkpoint_relative}
            request=service.files().create(body=body,media_body=media,fields='id')
        while result is None:
            status,result=request.next_chunk(num_retries=3)
            if status and checkpoint_relative is None:print(name,round(status.progress()*100),'% cargado')
    stored=service.files().get(fileId=result['id'],fields='id,name,size,md5Checksum,webViewLink').execute()
    assert int(stored['size'])==path.stat().st_size,'Drive no confirmó el tamaño completo.'
    assert stored['md5Checksum']==local_md5,'Drive no confirmó el checksum.'
    assert private_owner_only(stored['id']),'El archivo generado no está exclusivamente bajo el propietario.'
    return stored


def safe_checkpoint_path(relative):
    rel=PurePosixPath(relative)
    if rel.is_absolute() or '..' in rel.parts or not rel.parts or rel.suffix!='.json':
        raise ValueError('Ruta de checkpoint no admitida.')
    allowed={'source_manifest.json','order_report.json','model_selection.json','entrega_verificada.json'}
    if rel.parts[0] not in allowed and not rel.parts[0].startswith('trabajo_'):
        raise ValueError('El archivo no pertenece al trabajo recuperable.')
    return DURABLE.joinpath(*rel.parts)


def restore_checkpoints(folder_id,prefix='',depth=0):
    if depth>4:raise ValueError('Estructura de checkpoints demasiado profunda.')
    restored=0;token=None
    while True:
        page=service.files().list(q=f"'{folder_id}' in parents and trashed=false",pageToken=token,pageSize=1000,
            fields='nextPageToken,files(id,name,mimeType,size,md5Checksum,appProperties)').execute()
        for item in page.get('files',[]):
            name=item['name']
            if item['mimeType']=='application/vnd.google-apps.folder':
                # También recupera los checkpoints del notebook anterior, guardados con DriveFS.
                if (not prefix and name.startswith('trabajo_')) or (prefix.startswith('trabajo_') and name.startswith('checkpoints_')):
                    restored+=restore_checkpoints(item['id'],prefix+name+'/',depth+1)
                continue
            relative=item.get('appProperties',{}).get('vsaiRelPath') or (prefix+name)
            try:target=safe_checkpoint_path(relative)
            except ValueError:continue
            # Las nuevas copias de la API prevalecen sobre el árbol antiguo de DriveFS.
            if target.exists() and not item.get('appProperties',{}).get('vsaiRelPath'):continue
            blob=service.files().get_media(fileId=item['id']).execute(num_retries=3)
            assert len(blob)==int(item['size']),'Checkpoint incompleto en Drive.'
            if item.get('md5Checksum'):assert hashlib.md5(blob).hexdigest()==item['md5Checksum'],'Checksum de checkpoint incorrecto.'
            json.loads(blob) # comprobar formato antes de restaurar
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_suffix('.restore.tmp');temp.write_bytes(blob);os.replace(temp,target)
            restored+=1
        token=page.get('nextPageToken')
        if not token:break
    return restored


def persist_checkpoint(path):
    try:relative=Path(path).resolve().relative_to(DURABLE.resolve()).as_posix()
    except ValueError:return
    safe_checkpoint_path(relative)
    remote_name='checkpoint__'+relative.replace('/','__')
    try:upload_private_verified(path,'application/json',remote_name,relative)
    except HttpError as error:raise explain_drive_error(error,'guardar checkpoint privado') from None

restored=restore_checkpoints(remote_folder['id'])
CHECKPOINT_SAVER=persist_checkpoint
print('Checkpoints recuperados desde Drive:',restored)
print('Carpeta privada y recuperable: https://drive.google.com/drive/folders/'+remote_folder['id'])
required=sum(int(f['size']) for f in selected)*5+1024**3
assert shutil.disk_usage(LOCAL).free>required,'Bloqueo real: espacio local insuficiente.'
originals=[]
for info in selected:
    path=INPUT/info['name']
    valid=path.exists() and path.stat().st_size==int(info['size'])
    if valid and info.get('md5Checksum'):valid=checksum(path,'md5')==info['md5Checksum']
    if not valid:
        partial=path.with_suffix('.partial');partial.unlink(missing_ok=True)
        with open(partial,'wb') as f:
            downloader=MediaIoBaseDownload(f,service.files().get_media(fileId=info['id'],supportsAllDrives=True),chunksize=16*1024**2)
            done=False
            while not done:
                progress,done=downloader.next_chunk(num_retries=3)
                print(info['name'],round(progress.progress()*100),'% descargado')
        assert partial.stat().st_size==int(info['size']),'Tamaño descargado incorrecto.'
        if info.get('md5Checksum'):assert checksum(partial,'md5')==info['md5Checksum'],'Checksum de origen incorrecto.'
        validate(partial);os.replace(partial,path)
    originals.append(path)
atomic_json(DURABLE/'source_manifest.json',selected)
print('Tres copias verificadas. Los originales remotos no se modifican.')


In [ ]:
# 3 — Orden confirmado: aproximadamente 25 → 37 → 12 minutos, comprobado con FFprobe.
sources,ORDER_REPORT=automatic_order(originals)
atomic_json(DURABLE/'order_report.json',ORDER_REPORT)
print('Orden:',ORDER_REPORT['order'])
print('Criterio:',ORDER_REPORT['method'])
print('Duraciones reales, minutos:',ORDER_REPORT['actual_minutes'])
print('Se aplica el orden indicado por el usuario. La revisión editorial continúa pendiente.')
# No se crea ninguna apertura nueva: la prioridad son los tres originales completos.
INTRO=None


In [ ]:
# 4 — Fondo negro y recorte conservador de silencios, reutilizando la transcripción.
if globals().get('POLISH_ONLY',False):
    if 'RENDER' not in globals() or 'WORK' not in globals():
        raise RuntimeError('La sesión no conserva los archivos de la entrega anterior. Abrí el notebook actualizado y ejecutá todo para restaurarlos desde Drive.')
    source_joined=Path(RENDER)/'joined.mp4'
    if not source_joined.exists():raise RuntimeError('Falta el video limpio local. No se intenta editar sobre subtítulos ya quemados.')
    previous_report=dict(REPORT) if 'REPORT' in globals() else json.loads((WORK/'validation.json').read_text())
    master=Path(previous_report.get('source_words_file',WORK/'words_source_timeline.json'))
    if not master.exists():master=WORK/'words.json'
    source_words=json.loads(master.read_text())
    source_clips=previous_report.get('source_clips',previous_report['clips'])
    ASR_RUNTIME.update(previous_report.get('asr_runtime',{}))
    FINAL,SRT,REPORT=render_ready_interview(source_joined,source_words,source_clips,RENDER,WORK,previous_report)
else:
    # 4 — Transcripción, unión y subtítulos permanentes. GPU compatible o CPU eficiente.
    MODEL_RECORD=DURABLE/'model_selection.json'
    _previous_recovery_flag=bool(globals().get('CHECKPOINTS_ONLY',False))
    # A completed previous delivery proves that ASR checkpoints should already exist.
    CHECKPOINTS_ONLY=_previous_recovery_flag or (DURABLE/'entrega_verificada.json').exists()
    if CHECKPOINTS_ONLY:
        if not MODEL_RECORD.exists():
            raise RuntimeError('Falta el registro del modelo. No se reinició ninguna transcripción.')
        saved_runtime=json.loads(MODEL_RECORD.read_text())
        backend=saved_runtime.get('engine');model=saved_runtime.get('model')
        if backend not in ('faster-whisper','openai-whisper') or model not in ('base','small'):
            raise RuntimeError('No se pudo identificar el motor de los checkpoints; no se reinició la transcripción.')
        engine=backend+':'+model
        ASR_RUNTIME.update(saved_runtime)
        def infer(audio):
            raise RuntimeError('Falta un intervalo guardado. El modo de recuperación no vuelve a transcribir; conserva los avances existentes.')
        print('RECUPERACIÓN: solo checkpoints existentes. Whisper no se carga ni se ejecuta.')
    else:
        preferred=json.loads(MODEL_RECORD.read_text()).get('model') if MODEL_RECORD.exists() else None
        if preferred not in ('base','small'):preferred=None
        infer,engine=load_asr(model_name=preferred,allow_openai_fallback=True)
        atomic_json(MODEL_RECORD,dict(ASR_RUNTIME))
    print('Motor seleccionado:',engine,ASR_RUNTIME)
    ENGINE_KEY=re.sub(r'[^A-Za-z0-9_-]','_',engine)
    BASE_WORK=DURABLE/('trabajo_'+ENGINE_KEY)
    BASE_LOCAL=LOCAL/'render'/fingerprint/ENGINE_KEY
    REORDER_REUSE=prepare_reorder_reuse(sources,BASE_LOCAL,BASE_WORK,engine)
    WORK=BASE_WORK/REORDER_REUSE['namespace'] if REORDER_REUSE else BASE_WORK
    RENDER=BASE_LOCAL/REORDER_REUSE['namespace'] if REORDER_REUSE else BASE_LOCAL
    if REORDER_REUSE:
        print('CORRECCIÓN DE ORDEN: se conservan los checkpoints originales y se recalculan sus tiempos.')
    try:
        FINAL,SRT,REPORT=pipeline(sources,RENDER,WORK,infer,engine,intro=INTRO,reuse=REORDER_REUSE)
    finally:
        CHECKPOINTS_ONLY=_previous_recovery_flag
REPORT['continuity']=ORDER_REPORT
REPORT['asr_runtime']=dict(ASR_RUNTIME)
REPORT['human_sync_review']=False
REPORT['human_editorial_review']=False
REPORT['automatic_checks']={'three_sources_included':len(REPORT['clips'])==3,
    'decode_complete':REPORT['decode_complete'],
    'duration_matches_sum':abs(REPORT['final_seconds']-REPORT['expected_seconds'])<=.25,
    'subtitle_overlay_frame_checks':bool(REPORT['subtitle_frame_checks']),
    'drive_source_size_checked':True,'drive_source_md5_checked':all(f.get('md5Checksum') for f in selected),
    'recognized_words_preserved':REPORT['recognized_words_preserved'],
    'silence_vad_checked':REPORT['silence_edit']['vad_checked']}
REPORT['human_reviews_pending']=['continuidad semántica entre grabaciones','sincronía perceptual de voz y subtítulos',
    'nombres propios, fechas y lugares','fidelidad de reconocimiento y naturalidad de voces','lista y ubicación de cortes de silencio','descarga efectiva en el celular']
# Contexto textual en cada unión: solo palabras reconocidas, sin corregir ni inventar referencias.
words=json.loads(Path(REPORT['words_file']).read_text())
REPORT['boundary_contexts']=[]
for clip in REPORT['clips'][1:]:
    at=clip['output_start']
    REPORT['boundary_contexts'].append({'at':at,
        'before':' '.join(w['word'].strip() for w in words if at-12<=w['start']<at),
        'after':' '.join(w['word'].strip() for w in words if at<=w['start']<at+12)})
# Muestras reales de las uniones, para revisar después; no bloquean la entrega si fallan.
REVIEW_SAMPLES=[];REPORT['sample_warnings']=[]
for i,clip in enumerate(REPORT['clips'][1:],1):
    at=clip['output_start'];sample=LOCAL/f'union_{i}_ajustes_revision.mp4'
    try:
        sample.unlink(missing_ok=True)
        run(['ffmpeg','-v','error','-n','-ss',str(max(0,at-5)),'-i',FINAL,'-t','10',
             '-vf','scale=640:-2','-c:v','libx264','-preset','veryfast','-crf','26',
             '-c:a','aac','-b:a','96k','-threads','2',sample])
        validate(sample);REVIEW_SAMPLES.append(sample)
    except Exception as error:
        REPORT['sample_warnings'].append({'union':i,'error_type':type(error).__name__})
atomic_json(WORK/'validation.json',REPORT)
print('MP4 decodificado completamente:',REPORT['final_seconds'],'s. Subtítulos:',REPORT['subtitles_count'])
print('Continúa automáticamente la carga en Drive. No se requiere escribir SI.')


In [ ]:
# 5 — Entrega automática: archivos privados, verificación de servidor y enlaces móviles.
assert private_owner_only(remote_folder['id']),'No se carga material en una carpeta compartida.'
def upload_verified(path,mimetype):
    try:return upload_private_verified(path,mimetype)
    except HttpError as error:raise explain_drive_error(error,'cargar entregable final') from None

links=[]
for path,mime in [(FINAL,'video/mp4'),(SRT,'application/x-subrip')]:
    stored=upload_verified(path,mime);links.append(stored)
    print('CARGA VERIFICADA:',stored['name'],stored['webViewLink'])
REPORT['drive_uploads_verified']=[{'name':x['name'],'bytes':int(x['size']),'md5':x['md5Checksum'],'url':x['webViewLink']} for x in links]
REPORT['review_sample_links']=[]
for sample in REVIEW_SAMPLES:
    try:
        stored=upload_verified(sample,'video/mp4');REPORT['review_sample_links'].append(stored['webViewLink'])
    except Exception as error:
        REPORT['sample_warnings'].append({'sample':sample.name,'upload_error_type':type(error).__name__})
report_path=LOCAL/'ENTREVISTA_MALVINAS_INFORME_AJUSTES.txt'
lines=['ENTREVISTA DE MALVINAS — INFORME AUTOMÁTICO','',
    'Fuentes utilizadas (orden): '+', '.join(ORDER_REPORT['order']),
    'Orden: '+ORDER_REPORT['method'],
    'Continuidad semántica: pendiente de revisión humana.',
    f"Duración final: {REPORT['final_seconds']:.3f} s; esperada: {REPORT['expected_seconds']:.3f} s.",
    'Verificaciones automáticas: '+json.dumps(REPORT['automatic_checks'],ensure_ascii=False),
    'Motor local: '+json.dumps(REPORT['asr_runtime'],ensure_ascii=False),
    'MP4 y SRT: carga completa, tamaños y checksum MD5 verificados en Drive.',
    'Revisión humana: NO realizada. Pendiente: '+', '.join(REPORT['human_reviews_pending']),
    'Estilo: '+REPORT['subtitle_style'],
    'Silencios: mínimo 15 s; márgenes de 1,5 s; protección por palabras y detección de voz.',
    f"Recortes: {len(REPORT['silence_edit']['cuts'])}; retirados: {REPORT['silence_edit']['removed_seconds']:.3f} s.",
    'Palabras reconocidas conservadas: '+str(REPORT['recognized_words_preserved']),
    'Advertencias de detección de voz: '+str(REPORT['silence_edit']['vad_error']),
    'Advertencias ASR/lectura: '+str(len(REPORT['review'])),
    'Checkpoints JSON privados guardados y comprobados mediante Drive API.','',
    'Resultados:']
lines += [x['name']+': '+x['webViewLink'] for x in links]
def clock_label(seconds):
    value=max(0,int(round(seconds)))
    return f'{value//3600:02d}:{(value%3600)//60:02d}:{value%60:02d}'
lines += ['Cortes en la línea temporal anterior (HH:MM:SS):']
lines += [f"{clock_label(cut['source_start'])}–{clock_label(cut['source_end'])}; retirados {cut['removed_seconds']:.1f} s; unión nueva {clock_label(cut['new_time'])}; archivos: {', '.join(cut['sources'])}" for cut in REPORT['silence_edit']['cuts']]
lines += ['Muestras de continuidad para revisar después:']+REPORT['review_sample_links']
if REPORT['sample_warnings']:lines+=['Advertencias de muestras: '+json.dumps(REPORT['sample_warnings'],ensure_ascii=False)]
report_path.write_text('\n'.join(lines),encoding='utf-8')
report_link=upload_verified(report_path,'text/plain')
atomic_json(WORK/'validation.json',REPORT)
atomic_json(DURABLE/'entrega_verificada.json',{'files':links+[report_link],'validation':REPORT})
print('ENTREGA CARGADA Y VERIFICADA EN DRIVE PRIVADO')
from IPython.display import display,Markdown
for item in links+[report_link]:display(Markdown('['+item['name']+']('+item['webViewLink']+')'))
print('Carpeta: https://drive.google.com/drive/folders/'+remote_folder['id'])
print('Android: abrí el MP4 en Drive → menú ⋮ → Descargar.')
print('iPhone: Drive → menú ⋮ → Enviar una copia/Abrir en → Guardar en Archivos o Guardar video.')
print('El proceso verificó la carga en Drive; la descarga en el celular y la revisión editorial siguen pendientes.')


## Material académico — tarea independiente y opcional
La celda siguiente solo define herramientas. No pide archivos ni ejecuta trabajo durante «Ejecutar todo». Permite generar un PDF y una carátula privada a partir de un DOCX proporcionado por el usuario. No contiene nombres, preguntas ni documentos personales.


In [ ]:
import shutil
"""Build a private academic PDF and separate title clip from a local DOCX.
No student names, credentials or source file contents are embedded in this code.
"""
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as E
from xml.sax.saxutils import escape
import hashlib
import json
import re
import subprocess


def read_source(path):
    ns={'w':'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    with ZipFile(path) as z:
        xml=z.read('word/document.xml')
    if len(xml)>5*1024**2:raise ValueError('El XML excede el tamaño esperado para este trabajo.')
    root=E.fromstring(xml)
    def visible(node):
        tag=node.tag.rsplit('}',1)[-1]
        if tag in ('del','instrText'):return ''
        if tag=='t':return node.text or ''
        if tag in ('tab','br'):return ' '
        return ''.join(visible(child) for child in node)
    paragraphs=[' '.join(visible(p).split()) for p in root.findall('.//w:body/w:p',ns)]
    paragraphs=[p for p in paragraphs if p]
    university=next(p for p in paragraphs if p.startswith('UNIVERSIDAD'))
    course=next(p.split(':',1)[1].strip().rstrip('.') for p in paragraphs if p.startswith('MATERIA:'))
    course=course.title().replace('Publico','Público')
    commission=next(p.split(':',1)[1].strip().rstrip('.') for p in paragraphs if p.startswith(('COMISIÓN:','COMISION:')))
    a=next(i for i,p in enumerate(paragraphs) if p.startswith('ALUMNOS:'))+1
    b=next(i for i,p in enumerate(paragraphs) if p.startswith('LISTADO DE PREGUNTAS'))
    names=[p.rstrip('.') for p in paragraphs[a:b]]
    raw_questions=paragraphs[b+1:]
    questions=[];edits=[]
    for i,raw in enumerate(raw_questions,1):
        text=re.sub(r'^\s*\d+\s*[-.)]\s*','',raw).strip()
        text=re.sub(r'\s+-\s*$','',text)
        if i==1:
            text=text.replace('y que pensó','y qué pensó').replace('y que sintió','y qué sintió')
        if i==4 and not text.startswith('¿'):text='¿'+text
        questions.append(text)
        edits.append({'id':f'q{i:02}','original':raw,'edited':text,'level':1,
                      'note':'Numeración uniforme y ajuste mecánico de espacios, tildes interrogativas y signos; sin cambios de contenido.'})
    if len(names)!=7 or len(questions)!=10:
        raise ValueError(f'Se esperaban 7 integrantes y 10 preguntas: se encontraron {len(names)} y {len(questions)}.')
    return {'university':university,'course':course,'commission':commission,'names':names,
            'questions':questions,'raw_questions':raw_questions,'edits':edits,
            'source_sha256':hashlib.sha256(Path(path).read_bytes()).hexdigest()}


def fonts():
    import reportlab
    directory=Path(reportlab.__file__).parent/'fonts'
    return directory/'Vera.ttf',directory/'VeraBd.ttf'


def make_pdf(data,target):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.colors import HexColor
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.platypus import BaseDocTemplate,PageTemplate,Frame,Paragraph,Spacer,PageBreak,NextPageTemplate,Table,TableStyle,KeepTogether
    regular,bold=fonts()
    for name,file in [('Academic',regular),('Academic-Bold',bold)]:
        if name not in pdfmetrics.getRegisteredFontNames():pdfmetrics.registerFont(TTFont(name,str(file)))
    width,height=A4;left=23*mm;usable=width-46*mm
    ink=HexColor('#143844');gold=HexColor('#AA8749');gray=HexColor('#52616B')
    def cover(canvas,doc):
        canvas.saveState()
        canvas.setFillColor(ink);canvas.setFont('Academic-Bold',10)
        canvas.drawString(left,height-29*mm,data['university'])
        canvas.setStrokeColor(gold);canvas.setLineWidth(2);canvas.line(left,height-38*mm,left+38*mm,height-38*mm)
        canvas.setFillColor(gray);canvas.setFont('Academic',11);canvas.drawString(left,height-57*mm,'TRABAJO PRÁCTICO')
        canvas.setFillColor(ink);canvas.setFont('Academic-Bold',46);canvas.drawString(left,height-80*mm,'Malvinas')
        canvas.setFont('Academic',18);canvas.drawString(left,height-94*mm,'Guion de preguntas para la entrevista')
        canvas.setFont('Academic-Bold',13);canvas.drawString(left,height-119*mm,data['course'])
        canvas.setFillColor(gray);canvas.setFont('Academic',11);canvas.drawString(left,height-129*mm,'Comisión '+data['commission'])
        canvas.setFillColor(gold);canvas.setFont('Academic-Bold',10);canvas.drawString(left,height-154*mm,'INTEGRANTES')
        canvas.setFillColor(ink);canvas.setFont('Academic',12)
        for i,name in enumerate(data['names']):canvas.drawString(left,height-(166+i*8)*mm,name)
        canvas.setFillColor(gray);canvas.setFont('Academic',9);canvas.drawString(left,22*mm,'Entrevista documental · Trabajo práctico universitario')
        canvas.restoreState()
    def body(canvas,doc):
        canvas.saveState();canvas.setFillColor(gray);canvas.setFont('Academic',8)
        canvas.drawString(left,height-15*mm,'MALVINAS  /  GUION DE ENTREVISTA')
        canvas.setStrokeColor(HexColor('#D9DEDF'));canvas.setLineWidth(.5)
        canvas.line(left,height-19*mm,width-left,height-19*mm)
        canvas.drawString(left,12*mm,data['course']+' · Comisión '+data['commission'])
        canvas.drawRightString(width-left,12*mm,str(canvas.getPageNumber()-1));canvas.restoreState()
    doc=BaseDocTemplate(str(target),pagesize=A4,leftMargin=left,rightMargin=left,topMargin=28*mm,bottomMargin=21*mm,
                        title='Trabajo práctico: Malvinas — Guion de preguntas',author='; '.join(data['names']))
    frame=Frame(left,21*mm,usable,height-49*mm,leftPadding=0,rightPadding=0,topPadding=0,bottomPadding=0)
    doc.addPageTemplates([PageTemplate(id='cover',frames=[frame],onPage=cover),PageTemplate(id='body',frames=[frame],onPage=body)])
    heading=ParagraphStyle('heading',fontName='Academic-Bold',fontSize=18,leading=24,textColor=ink,spaceAfter=14)
    style=ParagraphStyle('question',fontName='Academic',fontSize=10.5,leading=14.3,textColor=HexColor('#172C34'),spaceAfter=0)
    number=ParagraphStyle('number',fontName='Academic-Bold',fontSize=10.5,leading=14.3,textColor=gold)
    story=[Spacer(1,1),NextPageTemplate('body'),PageBreak(),Paragraph('Listado de preguntas',heading)]
    required=38
    for i,text in enumerate(data['questions'],1):
        table=Table([[Paragraph(f'{i:02}',number),Paragraph(escape(text),style)]],colWidths=[25,usable-25])
        table.setStyle(TableStyle([('VALIGN',(0,0),(-1,-1),'TOP'),('LEFTPADDING',(0,0),(-1,-1),0),
                                  ('RIGHTPADDING',(0,0),(-1,-1),0),('TOPPADDING',(0,0),(-1,-1),0),('BOTTOMPADDING',(0,0),(-1,-1),7)]))
        required+=table.wrap(usable,10000)[1];story.append(KeepTogether([table]))
    if required>height-49*mm:raise ValueError(f'El cuestionario excede la página prevista: {required:.1f} pt.')
    doc.build(story)
    return {'body_height_points':round(required,1),'available_height_points':round(height-49*mm,1)}


def make_card(data,target):
    from PIL import Image,ImageDraw,ImageFont
    regular,bold=fonts()
    im=Image.new('RGB',(1920,1080),'#102936');d=ImageDraw.Draw(im)
    white='#F6F2E8';gold='#D3B16A';muted='#B5C6CC'
    def draw(text,x,y,size=32,boldface=False,color=white):
        f=ImageFont.truetype(str(bold if boldface else regular),size)
        d.text((x,y),text,font=f,fill=color)
    d.rectangle((140,145,260,151),fill=gold)
    draw(data['university'],140,82,30,True,color=muted)
    draw('TRABAJO PRÁCTICO',140,192,27,color=gold)
    draw('Malvinas',132,234,116,True)
    draw('Entrevista documental',140,383,43)
    draw('INTEGRANTES',140,502,23,True,color=gold)
    for i,name in enumerate(data['names']):
        x=140 if i<4 else 1000;y=560+(i if i<4 else i-4)*68
        size=31
        while ImageFont.truetype(str(regular),size).getlength(name)>760 and size>25:size-=1
        draw(name,x,y,size)
    d.line((140,916,1780,916),fill='#45606C',width=2)
    draw(data['course'],140,956,29,True)
    draw('COMISIÓN '+data['commission'],1320,956,29,color=muted)
    im.save(target)


def make_clip(image,target,seconds=12):
    subprocess.run(['ffmpeg','-v','error','-n','-loop','1','-framerate','30','-i',str(image),
                    '-f','lavfi','-i','anullsrc=channel_layout=stereo:sample_rate=48000',
                    '-t',str(seconds),'-c:v','libx264','-preset','veryfast','-tune','stillimage','-crf','20',
                    '-pix_fmt','yuv420p','-c:a','aac','-b:a','128k','-threads','2','-movflags','+faststart',str(target)],check=True)
    subprocess.run(['ffmpeg','-v','error','-xerror','-i',str(target),'-f','null','-'],check=True)


def build_assets(source,out,artwork_path=None):
    out=Path(out);out.mkdir(parents=True,exist_ok=True);data=read_source(source)
    pdf=out/'PREGUNTAS_MALVINAS.pdf'
    png=out/('CARATULA_MALVINAS_ACTUALIZADA.png' if artwork_path else 'CARATULA_MALVINAS.png')
    mp4=out/('CARATULA_MALVINAS_12S_ACTUALIZADA.mp4' if artwork_path else 'CARATULA_MALVINAS_12S.mp4')
    for path in (pdf,png,mp4):
        if path.exists():raise FileExistsError('Ya existe un derivado: '+str(path))
    layout=make_pdf(data,pdf)
    if artwork_path:
        # Artwork stays in the user's private session; never embed personal images in this notebook.
        subprocess.run(['ffmpeg','-v','error','-n','-i',str(artwork_path),
            '-vf','scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:color=0x102936,setsar=1',
            '-frames:v','1','-threads','2',str(png)],check=True)
    else:make_card(data,png)
    make_clip(png,mp4)
    audit={'build_version':2,'artwork_sha256':hashlib.sha256(Path(artwork_path).read_bytes()).hexdigest() if artwork_path else None,
           'source_sha256':data['source_sha256'],'names':data['names'],'question_count':len(data['questions']),
           'changes':data['edits'],'layout':layout,'human_review':False,'reviewer':'Codex (IA)',
           'scope':'Presentación y correcciones mecánicas; preguntas y nombres cotejados con la fuente.',
           'outputs':[{ 'file':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in (pdf,png,mp4)]}
    (out/'REVISION_EDITORIAL.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2))
    return {'pdf':str(pdf),'png':str(png),'mp4':str(mp4),'audit':str(out/'REVISION_EDITORIAL.json')},data



def save_academic_delivery_to_drive(expected_source_sha,cache_root=None,prepared_card_sha=None):
    """Optional companion task; recover valid assets and ignore malformed cache shapes."""
    import importlib.util,sys,uuid
    if 'service' not in globals() or 'upload_private_verified' not in globals():
        raise RuntimeError('Primero debe estar autorizada la sesión de Drive del notebook de la entrevista.')
    folder_info=globals().get('remote_folder')
    if folder_info is not None:
        if not isinstance(folder_info,dict) or not isinstance(folder_info.get('id'),str):
            raise RuntimeError('No se identifica la carpeta privada de salida. No se modificó la entrevista.')
        if 'private_owner_only' in globals() and not private_owner_only(folder_info['id']):
            raise RuntimeError('La carpeta de salida no es privada. No se carga documentación personal.')
    missing=[]
    if importlib.util.find_spec('reportlab') is None:missing.append('reportlab==4.4.9')
    if importlib.util.find_spec('PIL') is None:missing.append('pillow>=10,<13')
    if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
    if not shutil.which('ffmpeg'):raise RuntimeError('Falta FFmpeg en esta sesión.')
    base=Path(cache_root or '/content/malvinas_academic');base.mkdir(parents=True,exist_ok=True)
    source=base/'ORIGINAL_TRABAJO_PRACTICO.docx'
    uploaded={}
    if not source.exists() or hashlib.sha256(source.read_bytes()).hexdigest()!=expected_source_sha:
        candidates=[]
        try:
            payload=service.files().list(q="trashed=false and name='TRABAJO PRACTICO MALVINAS.docx'",
                includeItemsFromAllDrives=True,supportsAllDrives=True,pageSize=100,
                fields='files(id,name,size,mimeType)').execute()
            # Google normally returns {"files": [...]}; tolerate a normalized list as well.
            candidates=payload.get('files',[]) if isinstance(payload,dict) else payload
            if not isinstance(candidates,(list,tuple)):candidates=[]
        except Exception as error:
            print('La búsqueda opcional del DOCX en Drive no estuvo disponible:',type(error).__name__)
        found=False
        for item in candidates:
            if not isinstance(item,dict):continue
            if item.get('mimeType')!='application/vnd.openxmlformats-officedocument.wordprocessingml.document':continue
            try:
                if int(item.get('size',0))>10*1024**2 or not isinstance(item.get('id'),str):continue
                blob=service.files().get_media(fileId=item['id'],supportsAllDrives=True).execute(num_retries=3)
            except Exception as error:
                print('No se pudo leer un candidato DOCX en Drive:',type(error).__name__);continue
            if isinstance(blob,(bytes,bytearray)) and hashlib.sha256(blob).hexdigest()==expected_source_sha:
                source.write_bytes(blob);found=True;break
        if not found:
            from google.colab import files
            print('Seleccioná el mismo DOCX que adjuntaste en el chat. Solo se preparan el PDF y la carátula.')
            uploaded=files.upload()
            if not isinstance(uploaded,dict):
                raise RuntimeError('Colab no devolvió archivos válidos. Seleccioná un solo DOCX y reintentá esta celda.')
            matches=[blob for name,blob in uploaded.items() if isinstance(name,str) and name.lower().endswith('.docx')
                     and isinstance(blob,(bytes,bytearray)) and hashlib.sha256(blob).hexdigest()==expected_source_sha]
            if len(matches)!=1:raise ValueError('El archivo elegido no coincide con el DOCX original recibido.')
            source.write_bytes(matches[0])
    artwork_path=None
    if prepared_card_sha is not None:
        if not isinstance(prepared_card_sha,str) or not re.fullmatch(r'[a-f0-9]{64}',prepared_card_sha):
            raise ValueError('La huella de la carátula debe ser SHA-256.')
        artwork_path=base/('CARATULA_PRIVADA_'+prepared_card_sha[:12]+'.png')
        if not artwork_path.is_file() or hashlib.sha256(artwork_path.read_bytes()).hexdigest()!=prepared_card_sha:
            from google.colab import files
            def artwork_matches(mapping):
                if not isinstance(mapping,dict):return []
                return [blob for name,blob in mapping.items() if isinstance(name,str) and name.lower().endswith('.png')
                        and isinstance(blob,(bytes,bytearray)) and hashlib.sha256(blob).hexdigest()==prepared_card_sha]
            choices=artwork_matches(uploaded)
            if not choices:
                print('Seleccioná CARATULA_MALVINAS_ACTUALIZADA.png, descargada del chat. La entrevista no se procesa.')
                choices=artwork_matches(files.upload())
            if len(choices)!=1:raise ValueError('La imagen no coincide con la carátula preparada. Seleccioná el PNG original descargado del chat.')
            artwork_path.write_bytes(choices[0])
    tag=expected_source_sha[:12]+('_card_'+prepared_card_sha[:12] if prepared_card_sha else '')
    folder=base/tag
    names={'pdf':'PREGUNTAS_MALVINAS.pdf',
           'png':'CARATULA_MALVINAS_ACTUALIZADA.png' if artwork_path else 'CARATULA_MALVINAS.png',
           'mp4':'CARATULA_MALVINAS_12S_ACTUALIZADA.mp4' if artwork_path else 'CARATULA_MALVINAS_12S.mp4'}
    audit=folder/'REVISION_EDITORIAL.json'
    cached=False
    if audit.exists() and all((folder/name).is_file() for name in names.values()):
        try:
            state=json.loads(audit.read_text())
            records=state.get('outputs') if isinstance(state,dict) else None
            cached=(isinstance(state,dict) and state.get('source_sha256')==expected_source_sha
                    and state.get('build_version')==2 and state.get('artwork_sha256')==prepared_card_sha
                    and isinstance(records,list) and len(records)==3
                    and all(isinstance(row,dict) for row in records)
                    and {row.get('file') for row in records}==set(names.values()))
            if cached:
                cached=all(isinstance(row.get('sha256'),str) and hashlib.sha256((folder/row['file']).read_bytes()).hexdigest()==row['sha256'] for row in records)
        except (ValueError,TypeError,KeyError,OSError):cached=False
    if cached:
        print('PDF y carátula existentes verificados; se reutilizan.')
        paths={key:str(folder/name) for key,name in names.items()}
    else:
        # Preserve an incomplete/malformed prior attempt; rebuild only these small companion assets.
        if folder.exists():folder=base/(tag+'_'+uuid.uuid4().hex[:6])
        print('Preparando únicamente PDF y carátula; la entrevista no se procesa.')
        paths,_=build_assets(source,folder,artwork_path=artwork_path)
    from IPython.display import display,Markdown
    result={}
    for key,mime in [('pdf','application/pdf'),('png','image/png'),('mp4','video/mp4')]:
        stored=upload_private_verified(Path(paths[key]),mime)
        if not isinstance(stored,dict) or not isinstance(stored.get('name'),str) or not isinstance(stored.get('webViewLink'),str):
            raise RuntimeError('No se recibió confirmación válida de la carga del archivo complementario.')
        result[key]=stored
        display(Markdown('['+stored['name']+']('+stored['webViewLink']+')'))
    print('PDF y carátula cargados en tu carpeta privada, con tamaño y checksum verificados.')
    print('El clip de 12 segundos queda separado; la entrevista no se vuelve a renderizar.')
    return result


## Añadir la apertura a una entrevista ya terminada
Paso final opcional. La celda siguiente solo define funciones; no se ejecuta durante «Ejecutar todo». Reutiliza el MP4 y el SRT terminados de la misma sesión, adapta únicamente la apertura corta y copia el video y el audio de la entrevista sin recodificarlos. Conserva todos los fotogramas, paquetes de audio y subtítulos incrustados; desplaza el SRT editable por el tiempo real medido de la unión. Genera versiones `CON_APERTURA` y las verifica y carga en la carpeta privada de Drive. No inicia Whisper, no cambia el orden, los cortes ni el estilo y no sobrescribe el par anterior. La revisión humana final se informa como pendiente.


In [ ]:
"""Optional last step: prepend a private opening to an already finished interview.
Definitions only. No ASR, source editing, dependencies or media embedded in Git.
"""

def _opening_hash(path,algorithm='sha256'):
    import hashlib
    value=hashlib.new(algorithm)
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(4*1024**2),b''):value.update(chunk)
    return value.hexdigest()


def _opening_json(path,data):
    import json,os
    from pathlib import Path
    path=Path(path);tmp=path.with_suffix('.partial.json')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8');os.replace(tmp,path)


def _opening_run(args):
    import subprocess
    process=subprocess.run([str(x) for x in args],capture_output=True,text=True)
    if process.returncode:raise RuntimeError(process.stderr[-4000:])
    return process.stdout


def _opening_probe(path):
    import json
    return json.loads(_opening_run(['ffprobe','-v','error','-show_streams','-show_format','-of','json',path]))


def _opening_stream(info,kind):
    streams=[x for x in info['streams'] if x['codec_type']==kind]
    if len(streams)!=1:raise ValueError('Se esperaba una sola pista de '+kind+' en el video terminado.')
    return streams[0]


def _opening_progress(args,seconds,label):
    import subprocess,tempfile
    command=[str(x) for x in args]
    command[1:1]=['-nostdin','-progress','pipe:1','-nostats']
    with tempfile.TemporaryFile(mode='w+t') as errors:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=errors,text=True,bufsize=1)
        last=-10
        try:
            for line in process.stdout:
                if line.startswith('out_time_us='):
                    try:percent=max(0,min(99,int(int(line.split('=',1)[1])/1e6/max(seconds,.001)*100)))
                    except ValueError:continue
                    if percent>=last+10:print(label+': '+str(percent)+' %',flush=True);last=percent
            status=process.wait()
        except BaseException:
            process.terminate();process.wait();raise
        finally:process.stdout.close()
        if status:
            errors.seek(0);raise RuntimeError(errors.read()[-4000:])
    print(label+': 100 %',flush=True)


def _opening_shift_srt(text,offset_seconds):
    import re
    delta=round(offset_seconds*1000)
    pattern=re.compile(r'(?m)^(\d{2,}:\d{2}:\d{2}[,.]\d{3})([ \t]+-->[ \t]+)(\d{2,}:\d{2}:\d{2}[,.]\d{3})([^\r\n]*)(\r?)$')
    def value(stamp):
        h,m,s,ms=map(int,re.split(r'[:,.]',stamp));return ((h*60+m)*60+s)*1000+ms
    def stamp(ms):
        if ms<0:raise ValueError('Tiempo de subtítulo negativo.')
        return f'{ms//3600000:02d}:{ms//60000%60:02d}:{ms//1000%60:02d},{ms%1000:03d}'
    rows=[]
    def replace(match):
        start,end=value(match[1]),value(match[3])
        if end<=start:raise ValueError('El SRT de origen contiene un intervalo no válido.')
        rows.append((start+delta,end+delta))
        return stamp(start+delta)+match[2]+stamp(end+delta)+match[4]+match[5]
    output=pattern.sub(replace,text.lstrip('\ufeff'))
    if not rows:raise ValueError('El SRT de la entrevista no contiene intervalos reconocibles.')
    if any(b[0]<a[0] for a,b in zip(rows,rows[1:])):raise ValueError('El SRT no está en orden temporal.')
    return output,rows


def _opening_prefix_pts(path,count,kind='video'):
    import subprocess
    command=['ffprobe','-v','error','-select_streams','v:0' if kind=='video' else 'a:0','-show_packets',
             '-show_entries','packet=pts_time','-of','compact=p=0:nk=0',str(path)]
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,text=True)
    values=[]
    try:
        for line in process.stdout:
            fields=dict(part.split('=',1) for part in line.strip().split('|') if '=' in part)
            if 'pts_time' in fields:
                values.append(float(fields['pts_time']))
                if len(values)>=count:break
    finally:
        process.stdout.close()
        if process.poll() is None:process.terminate()
        process.wait()
    if len(values)<count:raise ValueError('No se localizaron todos los paquetes de video esperados.')
    return values


def _opening_count(info,kind,path):
    stream=_opening_stream(info,kind)
    try:return int(stream['nb_frames'])
    except (KeyError,ValueError,TypeError):
        import json
        data=json.loads(_opening_run(['ffprobe','-v','error','-select_streams','v:0' if kind=='video' else 'a:0',
            '-count_packets','-show_entries','stream=nb_read_packets','-of','json',path]))
        return int(data['streams'][0]['nb_read_packets'])


def _opening_pixels(path,seconds):
    import subprocess
    return subprocess.run(['ffmpeg','-v','error','-nostdin','-threads','2','-ss',f'{seconds:.6f}',
        '-i',str(path),'-map','0:v:0','-frames:v','1','-vf','scale=160:90,format=gray',
        '-f','rawvideo','-'],check=True,capture_output=True).stdout


def _opening_prepare(opening,interview_info,target):
    """Encode the short opening only; match the finished interview's stream properties."""
    from fractions import Fraction
    from pathlib import Path
    import math,os
    v=_opening_stream(interview_info,'video');a=_opening_stream(interview_info,'audio')
    if v['codec_name']!='h264' or a['codec_name']!='aac':
        raise ValueError('La copia rápida requiere el MP4 H.264/AAC generado por este notebook.')
    if v.get('pix_fmt') not in ('yuv420p','yuvj420p') or a.get('profile','LC')!='LC':
        raise ValueError('El formato del archivo no coincide con la entrega H.264/AAC estándar del notebook.')
    if any(abs(float(x.get('rotation',0)))%360>.01 for x in v.get('side_data_list',[])):
        raise ValueError('El video terminado conserva una rotación no esperada; no se modifica automáticamente.')
    width,height=int(v['width']),int(v['height'])
    if width%2 or height%2:raise ValueError('Se esperaba una resolución de dimensiones pares.')
    fps=Fraction(v.get('r_frame_rate') or v.get('avg_frame_rate'))
    if not 1<=float(fps)<=120:raise ValueError('No se pudo identificar la frecuencia de fotogramas.')
    opening_info=_opening_probe(opening);ov=_opening_stream(opening_info,'video')
    total=float(ov.get('duration',opening_info['format']['duration']))
    if not 0<total<=60:raise ValueError('La apertura debe durar menos de un minuto.')
    frames=max(1,math.floor(total*float(fps)+.5));seconds=frames/float(fps)
    sar=Fraction(v.get('sample_aspect_ratio','1:1').replace(':','/'))
    source_sar=Fraction(ov.get('sample_aspect_ratio','1:1').replace(':','/'))
    ratio=ov['width']*float(source_sar)/ov['height']/float(sar)
    scaled_width=max(2,min(width,math.floor(height*ratio)//2*2))
    scaled_height=max(2,min(height,math.floor(width/ratio)//2*2))
    full_range=v.get('color_range')=='pc' or v.get('pix_fmt')=='yuvj420p'
    out_range='pc' if full_range else 'tv'
    matrix=v.get('color_space')
    matrix=matrix if matrix in ('bt709','bt470bg','smpte170m','fcc','smpte240m') else 'bt601'
    scale=f'scale={scaled_width}:{scaled_height}:in_range=auto:out_range={out_range}:out_color_matrix={matrix}'
    vf=scale+f',pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:black,setsar={sar},fps={fps},setpts=PTS-STARTPTS,tpad=stop_mode=clone:stop_duration=1'
    profiles={'High':'high','Main':'main','Constrained Baseline':'baseline','Baseline':'baseline'}
    profile=profiles.get(v.get('profile'))
    if not profile:raise ValueError('El perfil H.264 de la entrevista no es compatible con esta copia rápida.')
    timebase=Fraction(v.get('time_base','1/15360'));timescale=timebase.denominator
    if timebase.numerator!=1:raise ValueError('Base de tiempo de video no compatible con la unión rápida.')
    channels=int(a.get('channels',2));rate=int(a.get('sample_rate',48000))
    if channels not in (1,2):raise ValueError('Se esperaba audio mono o estéreo en la entrevista.')
    target=Path(target);partial=target.with_suffix('.partial.mp4')
    if partial.exists():partial.rename(target.with_name(target.stem+'_intento_anterior_'+str(__import__('time').time_ns())+'.mp4'))
    args=['ffmpeg','-v','error','-n','-i',opening,'-map','0:v:0','-map','0:a:0','-vf',vf,
        '-af',f'aresample={rate}:async=1:first_pts=0,asetpts=PTS-STARTPTS,apad',
        '-t',f'{seconds:.9f}','-r',str(fps),'-vsync','cfr','-c:v','libx264','-preset','veryfast','-crf','20',
        '-profile:v',profile,'-pix_fmt','yuv420p','-color_range',out_range]
    if v.get('level',0)>0:args+=['-level:v',str(v['level']/10)]
    for field,flag in [('color_space','-colorspace'),('color_primaries','-color_primaries'),('color_transfer','-color_trc')]:
        value=v.get(field);args += [flag,value if value and value not in ('unknown','unspecified') else '2']
    args+=['-video_track_timescale',str(timescale),'-c:a','aac','-profile:a','aac_low','-b:a','128k',
        '-ar',str(rate),'-ac',str(channels),'-threads','2','-map_metadata','-1','-movflags','+faststart',partial]
    _opening_progress(args,seconds,'Adaptando solamente la apertura')
    result=_opening_probe(partial);rv=_opening_stream(result,'video');ra=_opening_stream(result,'audio')
    if (rv['width'],rv['height'],rv['codec_name'],rv['pix_fmt'])!=(width,height,v['codec_name'],v['pix_fmt']):
        raise ValueError('No se pudo adaptar la apertura al formato de la entrevista.')
    if (ra['codec_name'],ra['sample_rate'],ra['channels'])!=(a['codec_name'],a['sample_rate'],a['channels']):
        raise ValueError('No se pudo adaptar el audio de la apertura.')
    if abs(float(rv['duration'])-seconds)>.002 or abs(float(ra['duration'])-seconds)>.05:
        raise ValueError('La duración de la apertura adaptada no coincide con su audio.')
    os.replace(partial,target);return result


def _opening_join_local(opening,video,srt,folder):
    """Copy both interview streams, verify, then shift the separate editable SRT."""
    from pathlib import Path
    import json,os,shutil
    opening,video,srt,folder=map(Path,(opening,video,srt,folder));folder.mkdir(parents=True,exist_ok=True)
    if 'CON_APERTURA' in video.stem.upper():raise ValueError('Este archivo ya contiene una apertura; se evita agregarla dos veces.')
    signature={'version':3,'opening':_opening_hash(opening),'video':_opening_hash(video),'srt':_opening_hash(srt)}
    output=folder/'ENTREVISTA_MALVINAS_FINAL_CON_APERTURA.mp4'
    subtitles=folder/'ENTREVISTA_MALVINAS_SUBTITULOS_CON_APERTURA.srt'
    state_path=folder/'validacion_apertura.json'
    if state_path.is_file() and output.is_file() and subtitles.is_file():
        try:
            saved=json.loads(state_path.read_text())
            valid=(isinstance(saved,dict) and saved.get('signature')==signature and saved.get('decode_complete') is True
                and saved.get('video_sha256')==_opening_hash(output) and saved.get('srt_sha256')==_opening_hash(subtitles))
        except (ValueError,OSError):valid=False
        if valid:
            print('Unión ya terminada y verificada: se reutiliza sin procesar el video.',flush=True)
            return output,subtitles,saved
    info=_opening_probe(video);v=_opening_stream(info,'video');a=_opening_stream(info,'audio')
    source_seconds=float(info['format']['duration'])
    # Validate the editable subtitles before writing any new large output.
    _,original_rows=_opening_shift_srt(srt.read_text(encoding='utf-8-sig'),0)
    if original_rows[-1][1]>source_seconds*1000+250:raise ValueError('El SRT de origen excede la duración de la entrevista.')
    if shutil.disk_usage(folder).free<video.stat().st_size*1.05+100*1024**2:
        raise RuntimeError('Falta espacio para la nueva copia. Los archivos existentes se conservan.')
    normalized=folder/'apertura_adaptada.mp4'
    if normalized.exists():normalized.rename(folder/('apertura_previa_'+str(__import__('time').time_ns())+'.mp4'))
    prefix=_opening_prepare(opening,info,normalized)
    pv=_opening_stream(prefix,'video');pa=_opening_stream(prefix,'audio')
    prefix_seconds=float(pv['duration']);expected=prefix_seconds+source_seconds
    manifest=folder/'union.ffconcat'
    def quote(path):return str(path.resolve()).replace("'","'\\''")
    manifest.write_text("ffconcat version 1.0\nfile '"+quote(normalized)+f"'\nduration {prefix_seconds:.9f}\nfile '"+quote(video)+"'\n")
    partial=output.with_suffix('.partial.mp4')
    for old in (output,partial,subtitles):
        if old.exists():old.rename(old.with_name(old.stem+'_intento_anterior_'+str(__import__('time').time_ns())+old.suffix))
    print('Uniendo mediante copia de video y audio. No se transcribe ni se recodifica la entrevista.',flush=True)
    _opening_progress(['ffmpeg','-v','error','-n','-f','concat','-safe','0','-i',manifest,
        '-map','0:v:0','-map','0:a:0','-c','copy','-video_track_timescale',str(__import__('fractions').Fraction(v['time_base']).denominator),
        '-map_metadata','-1','-movflags','+faststart',partial],expected,'Copiando entrevista con apertura')
    result=_opening_probe(partial)
    actual=float(result['format']['duration'])
    if abs(actual-expected)>.15:raise ValueError('La duración de la unión no coincide con apertura más entrevista.')
    prefix_frames=_opening_count(prefix,'video',normalized);source_frames=_opening_count(info,'video',video)
    if _opening_count(result,'video',partial)!=prefix_frames+source_frames:raise ValueError('La unión no conserva todos los fotogramas.')
    if _opening_count(result,'audio',partial)!=_opening_count(prefix,'audio',normalized)+_opening_count(info,'audio',video):
        raise ValueError('La unión no conserva todos los paquetes de audio.')
    source_pts=_opening_prefix_pts(video,3);joined_pts=_opening_prefix_pts(partial,prefix_frames+3)[prefix_frames:]
    offsets=[b-a for a,b in zip(source_pts,joined_pts)];offset=sum(offsets)/len(offsets)
    if max(offsets)-min(offsets)>.0001 or abs(offset-prefix_seconds)>.1:
        raise ValueError('Los tiempos de la entrevista no conservaron un desplazamiento uniforme.')
    prefix_audio_frames=_opening_count(prefix,'audio',normalized)
    source_audio_pts=_opening_prefix_pts(video,3,kind='audio')
    joined_audio_pts=_opening_prefix_pts(partial,prefix_audio_frames+3,kind='audio')[prefix_audio_frames:]
    audio_offsets=[b-a for a,b in zip(source_audio_pts,joined_audio_pts)]
    if max(audio_offsets)-min(audio_offsets)>.03 or max(abs(x-offset) for x in audio_offsets)>.05:
        raise ValueError('La unión alteró el desplazamiento relativo de audio y video.')
    print('Verificando decodificación completa de la nueva copia.',flush=True)
    _opening_progress(['ffmpeg','-v','error','-xerror','-threads','2','-i',partial,'-map','0:v:0','-map','0:a:0','-f','null','-'],actual,'Verificación de integridad')
    samples=sorted(set(round(min(max(.1,x),max(.1,source_seconds-.1)),6) for x in (.51,source_seconds*.5+.013,source_seconds-.51)))
    for when in samples:
        before=_opening_pixels(video,when);after=_opening_pixels(partial,when+offset)
        if not before or before!=after:raise ValueError('La comparación de imagen detectó un cambio inesperado en la entrevista.')
    text,rows=_opening_shift_srt(srt.read_text(encoding='utf-8-sig'),offset)
    if rows[-1][1]>actual*1000+250:raise ValueError('El SRT ajustado excede la duración final.')
    subtitles.write_text(text,encoding='utf-8');os.replace(partial,output)
    report={'signature':signature,'source_filename':video.name,'source_seconds':source_seconds,
        'opening_seconds':prefix_seconds,'subtitle_shift_seconds':offset,'final_seconds':actual,
        'interview_video_encoding':'stream copy','interview_audio_encoding':'stream copy',
        'source_video_frames':source_frames,'opening_video_frames':prefix_frames,
        'decode_complete':True,'all_video_frames_preserved':True,'all_audio_packets_preserved':True,
        'audio_video_shift_difference_ms':round(max(abs(x-offset) for x in audio_offsets)*1000,3),
        'image_samples_unchanged':samples,'srt_cues_preserved':len(rows)==len(original_rows),
        'originals_modified':False,'asr_executed':False,'human_final_review':False,
        'human_reviews_pending':['reproducción y sincronía perceptual en la unión de apertura y testimonio'],
        'video_sha256':_opening_hash(output),'srt_sha256':_opening_hash(subtitles)}
    _opening_json(state_path,report);return output,subtitles,report


def append_opening_to_interview(opening_sha256,video_path=None,srt_path=None,cache_root=None):
    """One optional Colab cell: use the finished pair, acquire the small opening, upload a new pair."""
    from pathlib import Path
    import json,re,os
    if not isinstance(opening_sha256,str) or not re.fullmatch('[a-f0-9]{64}',opening_sha256):raise ValueError('Se requiere SHA-256 de la apertura completa.')
    folder=globals().get('remote_folder')
    if not isinstance(folder,dict) or not isinstance(folder.get('id'),str) or 'service' not in globals() or not callable(globals().get('upload_private_verified')):
        raise RuntimeError('Usá la sesión de Colab ya autorizada que generó la entrevista. No se reinició el procesamiento.')
    if not callable(globals().get('private_owner_only')) or not private_owner_only(folder['id']):
        raise RuntimeError('No se verificó una carpeta de Drive privada. No se cargan archivos.')
    base=Path(cache_root) if cache_root else Path(globals().get('LOCAL','/content/video_studio_malvinas'))/'apertura_final'
    base.mkdir(parents=True,exist_ok=True)
    def listing(name):
        escaped=name.replace('\\','\\\\').replace("'","\\'")
        data=service.files().list(q=f"'{folder['id']}' in parents and trashed=false and name='{escaped}'",
            fields='files(id,name,size,md5Checksum)',pageSize=100).execute()
        rows=data.get('files',[]) if isinstance(data,dict) else data
        if not isinstance(rows,list):raise RuntimeError('Drive no devolvió una lista de archivos válida.')
        rows=[r for r in rows if isinstance(r,dict) and isinstance(r.get('id'),str)]
        if len(rows)>1:raise RuntimeError('Hay archivos duplicados de '+name+'; no se elige una copia ambigua.')
        return rows
    def download(info,target):
        from googleapiclient.http import MediaIoBaseDownload
        target=Path(target)
        if target.is_file() and target.stat().st_size==int(info['size']) and _opening_hash(target,'md5')==info.get('md5Checksum'):return target
        part=target.with_suffix('.download.partial')
        with part.open('wb') as handle:
            worker=MediaIoBaseDownload(handle,service.files().get_media(fileId=info['id']),chunksize=16*1024**2);done=False;last=-10
            while not done:
                status,done=worker.next_chunk(num_retries=3)
                percent=round(status.progress()*100)
                if percent>=last+10:print(info['name'],percent,'% recuperado');last=percent
        if part.stat().st_size!=int(info['size']) or _opening_hash(part,'md5')!=info.get('md5Checksum'):
            raise ValueError('La descarga desde Drive no coincide con tamaño y checksum.')
        os.replace(part,target);return target
    video=Path(video_path) if video_path is not None else Path(globals()['FINAL']) if globals().get('FINAL') is not None else None
    subs=Path(srt_path) if srt_path is not None else Path(globals()['SRT']) if globals().get('SRT') is not None else None
    if not (video is not None and video.is_file() and subs is not None and subs.is_file()):
        pairs=[('ENTREVISTA_MALVINAS_FINAL_AJUSTADA.mp4','ENTREVISTA_MALVINAS_SUBTITULOS_AJUSTADOS.srt'),
               ('ENTREVISTA_MALVINAS_FINAL.mp4','ENTREVISTA_MALVINAS_SUBTITULOS.srt')]
        if video_path is not None or srt_path is not None:raise FileNotFoundError('Falta el par MP4/SRT indicado; no se transcribe de nuevo.')
        video=subs=None
        for video_name,sub_name in pairs:
            videos=listing(video_name)
            if videos:
                subtitles=listing(sub_name)
                if not subtitles:raise RuntimeError('Falta el SRT correspondiente a '+video_name+'. Se conservan los avances.')
                video=download(videos[0],base/video_name);subs=download(subtitles[0],base/sub_name);break
        if video is None:raise RuntimeError('No se encontró la entrevista terminada en la carpeta privada. No se ejecutó Whisper.')
    if 'CON_APERTURA' in video.stem.upper():raise ValueError('El archivo elegido ya tiene apertura.')
    print('Video terminado que se conservará:',video.name,flush=True)
    opening=base/'APERTURA_MALVINAS_CINE.mp4'
    if not opening.is_file() or _opening_hash(opening)!=opening_sha256:
        cached=next((p for p in Path('/content').glob('APERTURA_MALVINAS_CINE*.mp4') if p.is_file() and _opening_hash(p)==opening_sha256),None)
        if cached is not None:opening.write_bytes(cached.read_bytes())
        else:
            matches=listing(opening.name)
            if matches:download(matches[0],opening)
        if not opening.is_file() or _opening_hash(opening)!=opening_sha256:
            from google.colab import files
            print('Seleccioná APERTURA_MALVINAS_CINE.mp4 (17,3 segundos), descargada del chat.')
            uploaded=files.upload()
            blobs=[b for n,b in uploaded.items() if isinstance(n,str) and n.lower().endswith('.mp4') and isinstance(b,(bytes,bytearray)) and _opening_hash_bytes(b)==opening_sha256] if isinstance(uploaded,dict) else []
            if len(blobs)!=1:raise ValueError('El archivo elegido no coincide con la apertura completa preparada.')
            opening.write_bytes(blobs[0])
    upload_private_verified(opening,'video/mp4')
    job=_opening_hash_bytes(json.dumps([_opening_hash(video),_opening_hash(subs),opening_sha256]).encode())[:16]
    output,srt,report=_opening_join_local(opening,video,subs,base/('union_'+job))
    from IPython.display import display,Markdown
    links=[]
    for path,mime in [(output,'video/mp4'),(srt,'application/x-subrip')]:
        result=upload_private_verified(path,mime)
        if not isinstance(result,dict) or not isinstance(result.get('webViewLink'),str):raise RuntimeError('Falta confirmación de carga del archivo final.')
        links.append(result);display(Markdown('['+result['name']+']('+result['webViewLink']+')'))
    report_path=output.parent/'ENTREVISTA_MALVINAS_INFORME_APERTURA.txt'
    lines=['ENTREVISTA CON APERTURA — VERIFICACIÓN AUTOMÁTICA',
        'Entrevista reutilizada: '+video.name,
        f"Apertura adaptada: {report['opening_seconds']:.6f} s; desplazamiento del SRT: {report['subtitle_shift_seconds']:.6f} s.",
        f"Duración final: {report['final_seconds']:.6f} s.",
        'Entrevista: copia de video y audio, sin recodificación, sin cortes ni nueva transcripción.',
        'Subtítulos permanentes conservados; SRT editable desplazado a la nueva línea temporal.',
        'Verificaciones: decodificación completa, suma de duraciones, todos los fotogramas y paquetes de audio, muestras de imagen idénticas.',
        'Revisión humana final de la unión y sincronía: PENDIENTE.',
        'Cargas verificadas por tamaño, checksum MD5 y permisos privados mediante la API de Drive.']
    lines += [x['name']+': '+x['webViewLink'] for x in links]
    report_path.write_text('\n'.join(lines)+'\n',encoding='utf-8')
    stored=upload_private_verified(report_path,'text/plain');links.append(stored)
    display(Markdown('['+stored['name']+']('+stored['webViewLink']+')'))
    upload_private_verified(output.parent/'validacion_apertura.json','application/json',remote_name='validacion_apertura_'+job+'.json')
    print('ENTREVISTA CON APERTURA CARGADA Y VERIFICADA EN DRIVE PRIVADO')
    print('Carpeta: https://drive.google.com/drive/folders/'+folder['id'])
    print('Android: Drive → menú ⋮ del MP4 → Descargar. iPhone: Enviar una copia → Guardar en Archivos.')
    print('La revisión humana final sigue pendiente. La versión anterior y los originales se conservan.')
    return {'files':links,'validation':report}


def _opening_hash_bytes(data):
    import hashlib
    return hashlib.sha256(data).hexdigest()


## Revisión final por fragmentos, con auditoría privada
La celda siguiente solo define herramientas opcionales. Recibe una auditoría privada de IA; sustituye únicamente frases con una coincidencia textual normalizada, exacta y única cerca del tiempo informado. Las dudas y coincidencias ausentes se registran como pendientes. Muestra preguntas breves al comenzar las respuestas. Recodifica solamente los grupos de fotogramas afectados y copia los demás; mantiene el audio original sin recodificación. Verifica todos los tiempos de video y compara los paquetes y tiempos del audio. El área de subtítulos se reemplaza con fondo negro solo durante las correcciones. Guarda SRT e informe antes del video y conserva la entrega previa. No equivale a una escucha humana ni publica el contenido de la auditoría.


In [ ]:
"""Apply a private external subtitle audit and short question labels to an existing delivery.
No interview text, audit payload, names or transfer credentials are embedded here.
"""

def _audit_seconds(stamp):
    import re
    if not isinstance(stamp,str) or not re.fullmatch(r'\d{2,}:\d{2}:\d{2}',stamp):raise ValueError('Horario HH:MM:SS no válido.')
    h,m,s=map(int,stamp.split(':'))
    if m>=60 or s>=60:raise ValueError('Minutos o segundos fuera de rango: '+stamp)
    return h*3600+m*60+s


def _audit_norm(word):
    import unicodedata
    return ''.join(c for c in unicodedata.normalize('NFKD',word.casefold()) if unicodedata.category(c)!='Mn')


def _audit_reflow(words,line_limit):
    import textwrap
    def wrapped(group):return textwrap.wrap(' '.join(w['text'] for w in group),line_limit,break_long_words=False,break_on_hyphens=False)
    groups=[];group=[]
    for word in words:
        if group and (len(wrapped(group+[word]))>2 or word['end']-group[0]['start']>5000 or word['start']-group[-1]['end']>700):
            groups.append(group);group=[]
        group.append(word)
    if group:groups.append(group)
    result=[]
    for group in groups:
        lines=wrapped(group)
        if len(lines)>2:raise ValueError('Una corrección no cabe en dos líneas.')
        result.append({'start':group[0]['start'],'end':group[-1]['end'],'text':'\n'.join(lines)})
    return result


def _audit_plan(cues,data,width,height,total):
    import re
    words=[];tokens=[]
    for index,cue in enumerate(cues):
        pieces=list(re.finditer(r'\S+',cue['text']));weights=sum(len(m[0]) for m in pieces) or 1;cursor=0
        for m in pieces:
            begin=cue['start']+(cue['end']-cue['start'])*cursor/weights;cursor+=len(m[0])
            end=cue['start']+(cue['end']-cue['start'])*cursor/weights
            wi=len(words);lex=list(re.finditer(r'[^\W_]+',m[0],re.UNICODE))
            words.append({'text':m[0],'start':round(begin),'end':round(end),'cue':index})
            for j,match in enumerate(lex):tokens.append({'text':_audit_norm(match[0]),'word':wi,'first':j==0,'last':j==len(lex)-1})
    replacements=[];occupied=set();statuses=[]
    for patch in data.get('patches',[]):
        record={'id':patch.get('id'),'old':patch.get('old'),'new':patch.get('new')}
        try:
            a,b=_audit_seconds(patch['start']),_audit_seconds(patch['end'])
            if b<=a or a>=total:raise ValueError('Intervalo fuera del video.')
        except (ValueError,KeyError,TypeError) as error:
            record.update(status='pending_invalid_time',reason=str(error));statuses.append(record);continue
        needle=[_audit_norm(x) for x in re.findall(r'[^\W_]+',patch['old'],re.UNICODE)]
        if not needle or not isinstance(patch.get('new'),str) or not patch['new'].strip():
            record['status']='pending_invalid_text';statuses.append(record);continue
        possible=[]
        for k,token in enumerate(tokens):
            word=words[token['word']]
            if not ((a-20)*1000<=word['start']<=(b+20)*1000) or token['text']!=needle[0]:continue
            seq=tokens[k:k+len(needle)]
            if [t['text'] for t in seq]!=needle:continue
            if not seq[0]['first'] or not seq[-1]['last']:continue
            first,last=seq[0]['word'],seq[-1]['word']
            if words[last]['end']>(b+20)*1000:continue
            possible.append((first,last))
        if len(possible)!=1:
            record.update(status='pending_no_unique_exact_match',matches=len(possible));statuses.append(record);continue
        first,last=possible[0]
        if occupied.intersection(range(first,last+1)):
            record['status']='pending_overlap';statuses.append(record);continue
        occupied.update(range(first,last+1))
        replacement={'first':first,'last':last,'text':patch['new'],'cue_first':words[first]['cue'],'cue_last':words[last]['cue']}
        replacements.append(replacement)
        record.update(status='applied_external_audit',actual_start=words[first]['start']/1000,actual_end=words[last]['end']/1000)
        statuses.append(record)
    groups=[]
    for item in sorted(replacements,key=lambda x:x['cue_first']):
        if groups and item['cue_first']<=groups[-1][1]+1 and cues[item['cue_first']]['start']-cues[groups[-1][1]]['end']<=800:groups[-1][1]=max(groups[-1][1],item['cue_last'])
        else:groups.append([item['cue_first'],item['cue_last']])
    by_first={x['first']:x for x in replacements};result=[];changed=[];masks=[];handled=set()
    limit=max(18,min(42,round(42*(width/height)/(16/9))))
    for first_cue,last_cue in groups:
        wi=[i for i,w in enumerate(words) if first_cue<=w['cue']<=last_cue];rebuilt=[];i=wi[0]
        while i<=wi[-1]:
            patch=by_first.get(i)
            if patch:
                begin,end=words[i]['start'],words[patch['last']]['end'];pieces=patch['text'].split();weight=sum(map(len,pieces));cursor=0
                for word in pieces:
                    start=begin+(end-begin)*cursor/weight;cursor+=len(word)
                    finish=begin+(end-begin)*cursor/weight
                    rebuilt.append({'text':word,'start':round(start),'end':max(round(start)+1,round(finish))})
                i=patch['last']+1
            else:rebuilt.append(words[i]);i+=1
        events=_audit_reflow(rebuilt,limit);result+=events;changed+=events
        masks.append([cues[first_cue]['start']/1000,cues[last_cue]['end']/1000])
        handled.update(range(first_cue,last_cue+1))
    result += [dict(cue) for i,cue in enumerate(cues) if i not in handled]
    result.sort(key=lambda x:(x['start'],x['end']))
    questions=[]
    for question in data.get('questions',[]):
        try:
            start=_audit_seconds(question['start']);end=min(total,start+float(question.get('seconds',10)))
            if end<=start or not 0<=start<total:raise ValueError('Rótulo fuera del video.')
            questions.append({'start':round(start*1000),'end':round(end*1000),'text':f"Pregunta {int(question['number'])}: "+str(question['text'])})
        except (ValueError,TypeError,KeyError):statuses.append({'question':question.get('number'),'status':'pending_invalid_question_time'})
    return {'cues':result,'changed':changed,'masks':masks,'questions':questions,'status':statuses,
        'applied':len(replacements),'pending':sum(x['status'].startswith('pending') for x in statuses)}


def _audit_packets(path,kind='video',hashes=False):
    args=['ffprobe','-v','error','-select_streams','v:0' if kind=='video' else 'a:0','-show_packets',
          '-show_entries','packet=pts_time,duration_time,flags'+(',data_hash' if hashes else ''),'-of','compact=p=0:nk=0']
    if hashes:args+=['-show_data_hash','sha256']
    rows=[]
    for line in _opening_run(args+[path]).splitlines():
        row=dict(p.split('=',1) for p in line.split('|') if '=' in p)
        if row.get('pts_time') not in (None,'N/A'):
            rows.append({'pts':float(row['pts_time']),'duration':float(row['duration_time']) if row.get('duration_time') not in (None,'N/A') else 0,
                         'key':'K' in row.get('flags',''),'hash':row.get('data_hash')})
    return rows


def _audit_ass(plan,path,width,height,offset,start,end):
    import pysubs2,textwrap
    subs=pysubs2.SSAFile();subs.info.update(PlayResX=str(width),PlayResY=str(height),WrapStyle='0')
    font=max(14,48*height/1080)
    subs.styles['Default']=pysubs2.SSAStyle(fontname='DejaVu Sans',fontsize=font,primarycolor=pysubs2.Color(255,255,255),
        outlinecolor=pysubs2.Color(0,0,0),backcolor=pysubs2.Color(0,0,0),borderstyle=1,outline=1,shadow=0,
        alignment=pysubs2.Alignment.BOTTOM_CENTER,marginl=round(width*.07),marginr=round(width*.07),marginv=round(height*.06))
    subs.styles['Question']=pysubs2.SSAStyle(fontname='DejaVu Sans',fontsize=max(12,38*height/1080),bold=True,
        primarycolor=pysubs2.Color(255,255,255),outlinecolor=pysubs2.Color(0,0,0,25),backcolor=pysubs2.Color(0,0,0,25),
        borderstyle=3,outline=max(2,6*height/1080),shadow=0,alignment=pysubs2.Alignment.TOP_CENTER,
        marginl=round(width*.08),marginr=round(width*.08),marginv=round(height*.035))
    for style,events in [('Default',plan['changed']),('Question',plan['questions'])]:
        for event in events:
            a,b=event['start']/1000,event['end']/1000
            if b<=start or a>=end:continue
            text=event['text'].replace('{','(').replace('}',')')
            if style=='Question':text='\n'.join(textwrap.wrap(text,max(36,round(64*(width/height)/(16/9))),break_long_words=False))
            subs.append(pysubs2.SSAEvent(start=max(0,round((a-offset)*1000)),end=max(1,round((b-offset)*1000)),
                text=text.replace('\n',r'\N'),style=style))
    subs.save(str(path))


def _audit_render(source,srt,data,folder,checkpoint=None):
    from pathlib import Path
    from fractions import Fraction
    import json,os,shutil,bisect,pysubs2
    source,srt,folder=map(Path,(source,srt,folder));folder.mkdir(parents=True,exist_ok=True)
    signature={'version':2,'video':_opening_hash(source),'srt':_opening_hash(srt),'audit':_opening_hash_bytes(json.dumps(data,sort_keys=True,ensure_ascii=False).encode())}
    output=folder/'ENTREVISTA_MALVINAS_REVISION.mp4';subfile=folder/'ENTREVISTA_MALVINAS_REVISION.srt';receipt=folder/'validacion_revision.json'
    if output.exists() and subfile.exists() and receipt.exists():
        try:
            cached=json.loads(receipt.read_text());valid=isinstance(cached,dict) and cached.get('signature')==signature and cached.get('decode_complete') is True and cached.get('video_sha256')==_opening_hash(output) and cached.get('srt_sha256')==_opening_hash(subfile)
        except (ValueError,OSError):valid=False
        if valid:return output,subfile,cached
    info=_opening_probe(source);v=_opening_stream(info,'video');a=_opening_stream(info,'audio');total=float(info['format']['duration'])
    if v['codec_name']!='h264' or a['codec_name']!='aac':raise ValueError('Se requiere la entrega H.264/AAC de este notebook.')
    width,height=v['width'],v['height'];timebase=Fraction(v['time_base']);tick=float(timebase)
    if timebase.numerator!=1:raise ValueError('Base de tiempo de video no admitida para la edición por fragmentos.')
    original=pysubs2.load(str(srt),encoding='utf-8-sig')
    cues=[{'start':x.start,'end':x.end,'text':x.plaintext} for x in original if not x.is_comment]
    plan=_audit_plan(cues,data,width,height,total)
    report={'signature':signature,'audit_source':data.get('source'),'timeline':data.get('timeline'),
        'corrections_applied':plan['applied'],'pending_count':plan['pending'],'corrections':plan['status'],
        'questions':plan['questions'],'other_pending':data.get('pending',[]),'human_audio_review':False,
        'external_ai_audit':True,'new_asr_executed':False,'originals_modified':False,
        'subtitle_method':'Sustitución opaca del área de subtítulos solo en los intervalos corregidos; el resto de la imagen se conserva.'}
    updated=pysubs2.SSAFile()
    for cue in plan['cues']:updated.append(pysubs2.SSAEvent(start=cue['start'],end=cue['end'],text=cue['text'].replace('\n',r'\N')))
    updated.save(str(subfile));_opening_json(folder/'plan_revision.json',report)
    if checkpoint is not None:checkpoint(subfile,folder/'plan_revision.json')
    print(f"Coincidencias exactas: {plan['applied']}; pendientes: {plan['pending']}; preguntas: {len(plan['questions'])}.",flush=True)
    targets=plan['masks']+[[q['start']/1000,q['end']/1000] for q in plan['questions']]
    if not targets:raise RuntimeError('Ninguna propuesta coincide con este SRT. Se guardó el informe y no se exporta un video con correcciones supuestas.')
    if shutil.disk_usage(folder).free<source.stat().st_size*2.3+200*1024**2:raise RuntimeError('Falta espacio para las copias temporales; los archivos se conservan.')
    packets=_audit_packets(source);source_pts=sorted(p['pts'] for p in packets);keys=sorted(p['pts'] for p in packets if p['key'])
    if not keys or abs(keys[0]-source_pts[0])>tick:raise ValueError('El video no comienza en un fotograma clave.')
    video_start=source_pts[0];video_end=max(p['pts']+p['duration'] for p in packets)
    windows=[]
    for start,end in targets:
        left=keys[max(0,bisect.bisect_right(keys,max(video_start,start-.1))-1)]
        ri=bisect.bisect_left(keys,end+.1);right=keys[ri] if ri<len(keys) else video_end
        windows.append([left,right])
    merged=[]
    for left,right in sorted(windows):
        if merged and left<=merged[-1][1]+tick:merged[-1][1]=max(right,merged[-1][1])
        else:merged.append([left,right])
    bounds=sorted({video_start,video_end,*[x for pair in merged for x in pair]})
    split=folder/'partes';split.mkdir(exist_ok=True)
    pieces=sorted(split.glob('p*.mp4'))
    if len(pieces)!=len(bounds)-1:
        if pieces:
            split.rename(folder/('partes_incompletas_'+str(__import__('time').time_ns())));split.mkdir();pieces=[]
        cuts=','.join(f'{t:.6f}' for t in bounds[1:-1]);args=['ffmpeg','-v','error','-n','-copyts','-i',source,'-map','0:v:0','-an','-c:v','copy','-avoid_negative_ts','disabled','-f','segment']
        if cuts:args+=['-segment_times',cuts,'-segment_time_delta','0.00001']
        else:args+=['-segment_time',str(total+1)]
        args+=['-reset_timestamps','1','-segment_format','mp4','-segment_format_options','movflags=+faststart:video_track_timescale='+str(timebase.denominator),split/'p%04d.mp4']
        _opening_progress(args,total,'Separando fragmentos sin recodificar')
        pieces=sorted(split.glob('p*.mp4'))
    if len(pieces)!=len(bounds)-1:raise ValueError('La división no coincide con los fotogramas clave previstos.')
    joined=[];cursor=0;encoded_seconds=0;frame_checks=[]
    for index,piece in enumerate(pieces):
        local_rows=_audit_packets(piece);local_pts=sorted(p['pts'] for p in local_rows);count=len(local_pts)
        expected_pts=source_pts[cursor:cursor+count]
        if not expected_pts or count!=len(expected_pts):raise ValueError('La división alteró la cantidad de fotogramas.')
        offset=expected_pts[0]-local_pts[0];start=expected_pts[0];end=source_pts[cursor+count] if cursor+count<len(source_pts) else video_end;cursor+=count
        if max(abs((x+offset)-y) for x,y in zip(local_pts,expected_pts))>max(2*tick,.000003):raise ValueError('La división alteró los tiempos de video.')
        needs=any(start<b-tick and end>a+tick for a,b in merged)
        selected=piece
        if needs:
            selected=folder/f'corregido_{index:04d}.mp4';ass=folder/f'corregido_{index:04d}.ass'
            _audit_ass(plan,ass,width,height,offset,start,end)
            masks=[[max(x,start)-offset,min(y,end)-offset] for x,y in plan['masks'] if start<y and end>x]
            filters=[]
            if masks:
                font=max(14,48*height/1080);top=max(0,int(height-round(height*.06)-font*2.9-max(4,12*height/1080)));bottom=min(height,int(height-height*.035))
                enable='+'.join(f'gte(t,{x:.6f})*lt(t,{y:.6f})' for x,y in masks)
                filters.append(f"drawbox=x={round(width*.05)}:y={top}:w={round(width*.9)}:h={bottom-top}:color=black:t=fill:enable='{enable}'")
            filters.append('ass='+str(ass.resolve()).replace('\\','/').replace(':',r'\:').replace("'",r"\'"))
            if not selected.exists():
                partial=selected.with_suffix('.partial.mp4');partial.unlink(missing_ok=True)
                args=['ffmpeg','-v','error','-n','-copyts','-i',piece,'-map','0:v:0','-an','-vf',','.join(filters),
                    '-vsync','0','-enc_time_base',str(timebase),'-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt',v.get('pix_fmt','yuv420p'),
                    '-threads','2','-video_track_timescale',str(timebase.denominator),'-avoid_negative_ts','disabled','-map_metadata','-1']
                for field,flag in [('color_space','-colorspace'),('color_primaries','-color_primaries'),('color_transfer','-color_trc')]:
                    value=v.get(field);args += [flag,value if value and value not in ('unknown','unspecified') else '2']
                args+=['-color_range','pc' if v.get('color_range')=='pc' else 'tv','-movflags','+faststart',partial]
                _opening_progress(args,end-start,f'Corrigiendo fragmento {index+1}/{len(pieces)}');os.replace(partial,selected)
            fixed_pts=sorted(p['pts'] for p in _audit_packets(selected))
            if len(fixed_pts)!=len(local_pts) or max(abs((x-fixed_pts[0])-(y-local_pts[0])) for x,y in zip(fixed_pts,local_pts))>max(2*tick,.000003):raise ValueError('La corrección alteró los fotogramas o su sincronía.')
            encoded_seconds+=end-start
        joined.append((selected,end-start));frame_checks.append({'part':index,'frames':count,'encoded':needs,'start':start,'end':end})
    if cursor!=len(source_pts):raise ValueError('Faltan fotogramas al terminar la división.')
    manifest=folder/'revision.ffconcat'
    manifest.write_text('ffconcat version 1.0\n'+''.join("file '"+str(p.resolve()).replace("'","'\\''")+f"'\nduration {d:.9f}\n" for p,d in joined))
    partial=output.with_suffix('.partial.mp4');partial.unlink(missing_ok=True)
    _opening_progress(['ffmpeg','-v','error','-n','-copyts','-itsoffset',f'{video_start:.9f}','-f','concat','-safe','0','-i',manifest,
        '-i',source,'-map','0:v:0','-map','1:a:0','-c','copy','-video_track_timescale',str(timebase.denominator),
        '-avoid_negative_ts','disabled','-map_metadata','-1','-movflags','+faststart',partial],total,'Uniendo la revisión con el audio original')
    out_rows=_audit_packets(partial);out_pts=sorted(p['pts'] for p in out_rows)
    if len(out_pts)!=len(source_pts) or max(abs(x-y) for x,y in zip(out_pts,source_pts))>max(2*tick,.000003):raise ValueError('La unión alteró la línea temporal del video. No se carga esa copia.')
    print('Verificando que todos los paquetes de audio sean idénticos.',flush=True)
    original_audio=_audit_packets(source,kind='audio',hashes=True);new_audio=_audit_packets(partial,kind='audio',hashes=True)
    if len(original_audio)!=len(new_audio) or any(x['hash']!=y['hash'] or abs(x['pts']-y['pts'])>2/int(a['sample_rate']) for x,y in zip(original_audio,new_audio)):
        raise ValueError('La unión no conserva exactamente el audio y sus tiempos. No se carga esa copia.')
    _opening_progress(['ffmpeg','-v','error','-xerror','-threads','2','-i',partial,'-map','0:v:0','-map','0:a:0','-f','null','-'],total,'Verificando integridad final')
    os.replace(partial,output)
    report.update(decode_complete=True,video_frames_preserved=True,video_pts_preserved=True,audio_packets_identical=True,audio_pts_preserved=True,
        encoded_seconds=round(encoded_seconds,3),total_seconds=total,parts=frame_checks,video_sha256=_opening_hash(output),srt_sha256=_opening_hash(subfile))
    _opening_json(receipt,report);return output,subfile,report


def apply_private_review_in_colab(data,cache_root=None):
    from pathlib import Path
    import json,os
    if data.get('timeline')!='CON_APERTURA' or data.get('timeline_confirmed_by_user') is not True:raise ValueError('La auditoría no identifica la línea temporal confirmada.')
    folder=globals().get('remote_folder')
    if not isinstance(folder,dict) or not callable(globals().get('private_owner_only')) or not private_owner_only(folder['id']):raise RuntimeError('Se requiere la sesión de Drive privada ya autorizada.')
    base=Path(cache_root) if cache_root else Path(globals().get('LOCAL','/content/video_studio_malvinas'))/'revision_final';base.mkdir(parents=True,exist_ok=True)
    def locate(name):
        payload=service.files().list(q=f"'{folder['id']}' in parents and trashed=false and name='{name}'",fields='files(id,name,size,md5Checksum)',pageSize=100).execute()
        items=payload.get('files',[]) if isinstance(payload,dict) else payload
        if not isinstance(items,list) or len(items)!=1 or not isinstance(items[0],dict):raise RuntimeError('No se encontró una sola copia de '+name)
        info=items[0]
        local=Path(globals().get('LOCAL','/content/video_studio_malvinas'))
        candidates=[base/name]+list(local.rglob(name))
        for path in candidates:
            if path.is_file() and path.stat().st_size==int(info['size']) and _opening_hash(path,'md5')==info['md5Checksum']:return path
        from googleapiclient.http import MediaIoBaseDownload
        target=base/name;partial=target.with_suffix('.download.partial')
        with partial.open('wb') as handle:
            worker=MediaIoBaseDownload(handle,service.files().get_media(fileId=info['id']),chunksize=16*1024**2);done=False;last=-10
            while not done:
                status,done=worker.next_chunk(num_retries=3);percent=round(status.progress()*100)
                if percent>=last+10:print(name,percent,'% recuperado',flush=True);last=percent
        if partial.stat().st_size!=int(info['size']) or _opening_hash(partial,'md5')!=info['md5Checksum']:raise ValueError('La copia recuperada no coincide con Drive.')
        os.replace(partial,target);return target
    video=locate('ENTREVISTA_MALVINAS_FINAL_CON_APERTURA.mp4');srt=locate('ENTREVISTA_MALVINAS_SUBTITULOS_CON_APERTURA.srt')
    key=_opening_hash_bytes(json.dumps([_opening_hash(video),_opening_hash(srt),data],sort_keys=True,ensure_ascii=False).encode())[:16]
    work=base/('revision_'+key)
    # A completed partial review stays local and in Drive even if the final upload disconnects.
    work.mkdir(parents=True,exist_ok=True)
    _opening_json(work/'auditoria_privada.json',data)
    upload_private_verified(work/'auditoria_privada.json','application/json',remote_name='auditoria_privada_'+key+'.json')
    def checkpoint(subtitles,plan):
        upload_private_verified(subtitles,'application/x-subrip')
        upload_private_verified(plan,'application/json',remote_name='plan_revision_'+key+'.json')
        print('SRT corregido e informe de coincidencias guardados en Drive; continúa el video.',flush=True)
    output,subtitles,report=_audit_render(video,srt,data,work,checkpoint=checkpoint)
    from IPython.display import display,Markdown
    links=[]
    for path,mime in [(subtitles,'application/x-subrip'),(output,'video/mp4')]:
        stored=upload_private_verified(path,mime);links.append(stored)
        display(Markdown('['+stored['name']+']('+stored['webViewLink']+')'))
    path=work/'ENTREVISTA_MALVINAS_INFORME_REVISION.txt'
    path.write_text('\n'.join([
        'REVISIÓN A PARTIR DE AUDITORÍA EXTERNA DE IA',
        f"Correcciones de coincidencia exacta aplicadas: {report['corrections_applied']}; pendientes: {report['pending_count']}.",
        f"Rótulos de preguntas: {len(report['questions'])}; diez segundos al inicio de cada respuesta.",
        f"Video recodificado solo por fragmentos: {report.get('encoded_seconds',0):.1f} s de {report.get('total_seconds',0):.1f} s.",
        'Audio: todos los paquetes y sus tiempos comparados e idénticos. Video: todos los fotogramas y sus tiempos conservados; decodificación completa verificada.',
        'El área de subtítulos se cubre con negro solo durante las correcciones; las zonas restantes conservan la imagen.',
        'No se eliminaron palabras ni se recortó el microcorte de Zoom. No se transcribió nuevamente.',
        'La auditoría externa es de IA. Revisión humana perceptual final: PENDIENTE.',
        'Pendientes adicionales: '+json.dumps(report['other_pending'],ensure_ascii=False),
        'Detalle de cada propuesta: '+json.dumps(report['corrections'],ensure_ascii=False),
        'Resultados:']+[x['name']+': '+x['webViewLink'] for x in links])+'\n',encoding='utf-8')
    stored=upload_private_verified(path,'text/plain');links.append(stored)
    display(Markdown('['+stored['name']+']('+stored['webViewLink']+')'))
    upload_private_verified(work/'validacion_revision.json','application/json',remote_name='validacion_revision_'+key+'.json')
    print('REVISIÓN CARGADA Y VERIFICADA EN DRIVE PRIVADO. La entrega anterior permanece intacta.',flush=True)
    return {'files':links,'validation':report}
